# VeriStamp · M1: frequency/noise ResNet-50 detector (complete pipeline)

**DATA 255 · Group 7.** This notebook trains and evaluates **M1** and an **RGB-only ResNet-50 baseline** on GenImage with a leave-one-generator-out split. It implements **only M1**: no M2, Fusion, M3 or demo.

```
Dataset inspection → Manifest → Duplicate check → Leakage-safe split → Shared preprocessing
      → RGB baseline + M1 (high-pass + FFT) → Sanity checks → Training + validation
      → Seen-generator test → Unseen-generator test → Metrics, plots → Checkpoints + results table
```

### How to use this notebook
**Runtime → Change runtime type → A100 GPU.** CPU works for Section 0 and the small checks, but is far too slow for training.

1. **Run Section 0: Dataset Acquisition and Setup.** Choose a workflow and a dataset folder in 0.1. Section 0 stops with a clear message wherever you need to act (pick 3 generators, extract data, confirm labels). Fix the setting, then run Section 0 again.
2. **Confirm the dataset summary and folder mapping:** check the mapping (0.7) and the sample images (0.9), then set `LABELS_CONFIRMED = True` in 0.11.
3. **Set `DATA_ROOT` from Section 0:** 0.11 prints the exact `DATA_ROOT` and `N_PER_CLASS` lines. Copy them into Section 1.
4. **Run Section 1: Project Configuration.**
5. **Continue with the existing M1 pipeline (Sections 2–22).** Section 4 checks that Section 0 verified the data. Section 5 reuses the mapping you confirmed in Section 0. In Section 7, choose `UNSEEN_GENERATOR`. Send back the outputs listed at the end of each section.

**Resuming after a disconnect:** Section 0 does not need to be rerun. Its result is saved as `DATASET_SETUP.json` in your dataset folder. Run Sections 1–3 again, then continue: the split, the preprocessed cache and the checkpoints are all reloaded from Drive. Rerun Section 0 only if you change the data, the generators or the subset level.

### Tags used in this notebook
| Tag | Meaning |
|---|---|
| 🟦 **Proposal** | required by our submitted proposal |
| 🟨 **Interim** | simplified for the intermediate submission |
| ⬜ **Optional** | extra, can be added later |

| Part | Status |
|---|---|
| Official GenImage source, 3 selected generators, labels confirmed by a person (Section 0) | 🟦 Proposal (dataset) |
| ResNet-50 (ImageNet) fine-tuned on high-pass + FFT inputs | 🟦 Proposal |
| Same preprocessing + JPEG re-encode for Real and AI (bias control) | 🟦 Proposal |
| Leave-one-generator-out, accuracy + AUROC on the held-out generator | 🟦 Proposal (all 3 folds) · 🟨 Interim: **one fold** |
| 3k–5k images per class per generator | 🟦 Proposal · 🟨 Interim: start at 200, then 500 |
| RGB-only ResNet-50 baseline trained on the same split | ⬜ addition: a controlled comparison |
| `rgb_hp_fft` (7-channel) mode | ⬜ Optional, supported by the code but not run by default |
| Grad-CAM, JPEG 65/30 degradation track, published baselines | 🟦 Proposal, but **after** the interim (not in this notebook) |

### Built for reuse
All reusable code is written once, in **Section 2**, into a small library (`vslib/`) that is saved to Drive.

| Module | Used by |
|---|---|
| `vs_common` (config, seeds, device, checkpoints, logs) | M1, M2, Fusion, **M3** |
| `vs_data` (scan, hash, duplicates) | GenImage now; the **separate COCO manifest for M3** later |
| `vs_genimage` (manifest + the one fixed split) | M1, M2, Fusion |
| `vs_preprocess` (bias-controlled preprocessing + cache) | M1, M2, Fusion. Outputs un-normalised 0..1 tensors, so M2 can apply CLIP normalisation to the *same* cache |
| `vs_metrics`, `vs_train`, `vs_plots` | M1, M2, Fusion (M3 adds its own watermark metrics later) |
| `vs_features`, `vs_models` | M1 and the RGB baseline |

M3's COCO manifest, split, 128×128 preprocessing and watermark metrics are **designed to reuse `vs_common` and `vs_data`**, but are **not implemented here**, as requested.

## 0 · GenImage Dataset Acquisition and Setup
**Purpose:** get the GenImage data into Google Drive, check it, and confirm which folders are which, **before** the M1 pipeline touches it. This section never trains anything and never builds train/val/test splits. Section 7 does the splitting later.

```
             ┌─ Workflow A: "existing_drive" ─ data already extracted in Drive ─────────────┐
 0.1 choose ─┼─ Workflow B: "download_instructions" ─ official links + manual steps, stop ──┼─► 0.5 inspect DATASET_DIR
   mode      └─ "download_subset" ─ extract ONLY a small sample of 3 generators from the ───┘     0.6 count + suggested mapping
                official zip archives in your Drive (single or split)                   0.7 confirm folder → generator/label
                                                                                                  0.8 formats, sizes, corrupt files
                                                                                                  0.9 sample images (Real vs AI)
                                                                                                  0.10 DATASET_SUMMARY.csv + DATASET_README.md
                                                                                                  0.11 VERIFY → DATASET_SETUP.json + DATA_ROOT for Section 1
```

**Official source:** <https://github.com/GenImage-Dataset/GenImage>. The README lists a Google Drive folder and a Baidu Netdisk mirror, and generator folders named Midjourney, VQDM, Wukong, Stable Diffusion V1.4, Stable Diffusion V1.5, GLIDE, BigGAN and ADM. Each has `train` and `val` sub-folders containing `ai` and `nature`. The README says each folder contains compressed files, but does not state the archive format. This section **detects** what you actually have and assumes nothing.

**Rules this section follows:**
- It never downloads the full million-image dataset.
- Generator folders are never merged or renamed.
- `ai` and `nature` stay separate.
- You confirm every label yourself.
- No passwords or API keys are needed.

### 0.1 · Settings ✏️
Choose a workflow, point `DATASET_DIR` at your data, and later pick 3 generators. Run Section 0 top to bottom. Wherever you need to act, it stops with a clear message: fix the setting in 0.1, then run Section 0 again.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 0 SETTINGS: edit the lines marked  <<< CHANGE                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
DATA_SOURCE_MODE = "existing_drive"     # <<< CHANGE
# Options:
#   "existing_drive"        = data is already downloaded AND extracted in your Google Drive (Workflow A)
#   "download_instructions" = show the official links and step-by-step instructions, then stop (Workflow B)
#   "download_subset"       = extract ONLY a small sample of the selected generators from the official
#                             zip archives (single or split .z01/.z02/...) visible in your Drive (see 0.4)

DATASET_DIR = "/content/drive/MyDrive/Datasets/GenImage"   # <<< CHANGE: the folder whose SUB-FOLDERS are the generators
SELECTED_GENERATORS = []        # <<< CHANGE after cell 0.5: 3 generator folder names, copied EXACTLY as printed

DEBUG_IMAGES_PER_CLASS = 200    # first debug run (per class, per generator)
INTERIM_IMAGES_PER_CLASS = 500  # interim run
FINAL_IMAGES_PER_CLASS = 3000   # final run (proposal: 3000-5000)
SUBSET_LEVEL = "debug"          # <<< CHANGE later: "debug" | "interim" | "final"

DOWNLOAD_DATE = ""              # <<< CHANGE: the date you downloaded the archives, e.g. "2026-10-26" (goes into DATASET_README.md)
ALLOW_ARCHIVES_IN_SELECTED = False   # set True ONLY if you deliberately keep .zip files inside the selected generator folders

ARCHIVE_SOURCE_DIR = ""         # <<< CHANGE (only for "download_subset"): Drive folder where the OFFICIAL generator folders are visible
SEED_S0 = 42                    # fixed seed for choosing which images to extract
CHECK_PER_FOLDER = 200          # images opened per folder in the corruption check (None = every file; slow on Drive)
# ════════════════════════════════════════════════════════════════════════════

SUBSET_SIZES = {"debug": DEBUG_IMAGES_PER_CLASS, "interim": INTERIM_IMAGES_PER_CLASS, "final": FINAL_IMAGES_PER_CLASS}
assert DATA_SOURCE_MODE in ("existing_drive", "download_instructions", "download_subset"), "Unknown DATA_SOURCE_MODE"
assert SUBSET_LEVEL in SUBSET_SIZES, f"SUBSET_LEVEL must be one of {list(SUBSET_SIZES)}"
TARGET_PER_CLASS = SUBSET_SIZES[SUBSET_LEVEL]
GENIMAGE_SOURCE = {
    "repository": "https://github.com/GenImage-Dataset/GenImage",
    "paper": "M. Zhu et al., GenImage: A Million-Scale Benchmark for Detecting AI-Generated Image, NeurIPS 2023 Datasets and Benchmarks",
    "google_drive": "https://drive.google.com/drive/folders/1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS?usp=sharing",
    "baidu_netdisk": "https://pan.baidu.com/s/1i0OFqYN5i6oFAxeK6bIwRQ (public share code 'ztf1', as published in the official README)",
    "license": "see the LICENSE file in the official repository",
}
print(f"Mode: {DATA_SOURCE_MODE} | dataset folder: {DATASET_DIR}")
print(f"Subset level: {SUBSET_LEVEL} -> {TARGET_PER_CLASS} images per class per generator")

### 0.2 · Connect Google Drive and show the official source

In [ ]:
import os, re, json, math, random, shutil, zipfile, datetime, time
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: skipping Google Drive mount.")

IMAGE_EXTS_S0 = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
ARCHIVE_RE = re.compile(r"\.(zip|z\d{2}|\d{3}|tar|tgz|gz|bz2|xz|rar|7z)$", re.IGNORECASE)   # archive-like file names
SETUP_DIRNAME = "_veristamp_setup"      # where Section 0 keeps its own files inside DATASET_DIR (ignored by M1)


class StopHere(Exception):
    """Raised on purpose to stop 'Run all' at a point where you need to act. Read the message."""


print("Official GenImage source:")
for k, v in GENIMAGE_SOURCE.items():
    print(f"  {k:>13}: {v}")

### 0.3 · Workflow B: how to get the data (manual, always works)
No passwords or API keys are needed.

**Step 1: open the official source.** Go to <https://github.com/GenImage-Dataset/GenImage> and follow its Google Drive link, or the Baidu Netdisk mirror (share code `ztf1`, published in the README). Read the repository's LICENSE before use.

**Step 2: choose 3 generators.** Each generator is its own folder. Only those 3 are needed. A suggestion to discuss as a team: one per model family, so that "unseen generator" really means a different kind of model. For example, one latent-diffusion folder (a Stable Diffusion version), one pixel-diffusion folder (ADM or GLIDE) and one GAN (BigGAN). The choice is yours.

**Step 3: bring only those 3 folders into your Drive.** Two options:
- **Shortcut (no download):** in the shared Google Drive folder, right-click a generator folder → **Organize → Add shortcut** → choose a folder in *My Drive*, for example `MyDrive/Datasets/GenImage_official`. The archives then appear in Colab under that path, without using your Drive space. Use this with `DATA_SOURCE_MODE = "download_subset"`.
- **Download:** download the archives for those 3 generators to your computer, then upload them to `MyDrive/Datasets/GenImage_archives/<generator name>/`. This is slow for large files. Google Drive may also refuse popular shared files for a while ("download quota exceeded"); if so, try later or use the Baidu mirror.

**Step 4: extract while keeping the folder names.** The goal is `DATASET_DIR/<generator>/.../ai/` and `.../nature/`, with one top-level folder per generator. Never mix generators.

| What you see in the generator folder | What to do |
|---|---|
| one or more single `name.zip` files | `download_subset` mode can extract a small sample directly (0.4). Or extract fully with `!unzip -q "<zip>" -d "<DATASET_DIR>/<generator>"` |
| split parts, e.g. `name.zip` + `name.z01`, `name.z02` … | `download_subset` mode reads a sample directly (0.4). Full extraction: rejoin first: `!zip -s 0 "name.zip" --out "/content/joined.zip"`, then extract only what you need: `!unzip -q "/content/joined.zip" "*/ai/*" "*/nature/*" -d "<DATASET_DIR>/<generator>"` |
| numbered parts, e.g. `name.zip.001`, `name.zip.002` … | rejoin: `!cat "name.zip".0* > "/content/joined.zip"`, then extract as above |
| `.tar` / `.tar.gz` | `!tar -xf "<file>" -C "<DATASET_DIR>/<generator>"` |
| something else (`.rar`, `.7z`) | `!apt-get -qq install p7zip-full` then `!7z x "<file>" -o"<DATASET_DIR>/<generator>"` |

**Disk space:** rejoining split archives needs room for the whole joined file. Write it to Colab's local disk (`/content/…`), not Drive, and delete it after extracting. Run `!df -h /content` to see free space.

**Step 5: run Section 0 again** with `DATA_SOURCE_MODE = "existing_drive"` and `DATASET_DIR` pointing at the folder that contains the 3 generator folders.

**For the debug run you do not need everything:** 200 Real + 200 AI images per generator is enough. Partially extracted folders are fine, as long as each generator keeps its own `ai` and `nature` folders.

In [ ]:
if DATA_SOURCE_MODE == "download_instructions":
    print("Follow the steps in the text cell above (0.3). In short:")
    print("  1. Open", GENIMAGE_SOURCE["repository"], "-> Google Drive link (or Baidu mirror).")
    print("  2. Pick 3 generator folders.")
    print("  3. Add them to your Drive as shortcuts, or download + upload their archives.")
    print("  4. Extract each into DATASET_DIR/<generator name>/ (keep ai/ and nature/ separate), or use 'download_subset'.")
    print("  5. Set DATA_SOURCE_MODE = 'existing_drive' (or 'download_subset') in 0.1 and run Section 0 again.")
    raise StopHere("Stopped on purpose: get the data first, then change DATA_SOURCE_MODE in cell 0.1.")
print(f"Mode is '{DATA_SOURCE_MODE}': download instructions skipped (they are in the text cell above if needed).")

### 0.4 · Optional: extract a small subset from the official archives (`download_subset` only)
**What it does:** for each selected generator, it looks inside `ARCHIVE_SOURCE_DIR/<generator>/` for archive files.
- **Zip archives, single (`name.zip`) or split into parts (`name.z01`, `name.z02`, … + `name.zip`, as on the official GenImage Drive):** these can be read selectively, **without joining the parts or downloading everything**. The last part holds the archive's table of contents, which says in which part and at which byte every image sits; the cell jumps straight to the chosen images. Each image's checksum is verified. For every image folder inside the zip (for example `…/train/ai`, `…/train/nature`, `…/val/ai`), it copies a random sample of `1.3 × TARGET_PER_CLASS` images into `DATASET_DIR/<generator>/`, keeping the original paths. The fixed seed means the same images are picked every time. The extra 30% replaces duplicates or broken files found later. Files already copied are skipped, so you can rerun it after a disconnect.
- **Other archives (`.rar`, `.7z`, `.tar`) or archives with a missing part:** these are reported, and you use the manual steps in 0.3.

**Speed:** reading the table of contents takes some seconds to a few minutes per generator (it is tens of MB). The images are then read 8 at a time.

It never extracts a whole generator. It records what was actually extracted in `DATASET_DIR/_veristamp_setup/subset_extraction_log.json`.

In [ ]:
import struct, zlib, bisect
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


class Member:
    """One file inside an archive: its path in the archive, sizes, and where its data starts."""
    def __init__(self, filename, file_size, compress_size, method, crc, flags, disk, offset):
        self.filename, self.file_size, self.compress_size = filename, file_size, compress_size
        self.method, self.crc, self.flags, self.disk, self.offset = method, crc, flags, disk, offset


class SplitZip:
    """Read single files out of a SPLIT zip (name.z01, name.z02, ..., name.zip) WITHOUT joining the parts.

    How: the last part (.zip) ends with the archive's table of contents (the "central directory").
    For every file it stores (part number, byte offset in that part). We read the table once,
    then jump straight to the few images we want. Only those bytes are read from Drive.
    """
    def __init__(self, parts):
        self.parts = [str(p) for p in parts]                      # in order: .z01, .z02, ..., .zip
        sizes = [os.path.getsize(p) for p in self.parts]
        self.starts = [0]
        for s in sizes:
            self.starts.append(self.starts[-1] + s)               # where each part begins in the joined stream
        self.total = self.starts[-1]
        self._read_central_directory()

    def _read(self, pos, n):
        """Read n bytes at position pos of the (virtual) joined stream; may cross part boundaries."""
        out = bytearray()
        while n > 0:
            i = bisect.bisect_right(self.starts, pos) - 1
            with open(self.parts[i], "rb") as f:
                f.seek(pos - self.starts[i])
                chunk = f.read(min(n, self.starts[i + 1] - pos))
            if not chunk:
                raise IOError("unexpected end of archive")
            out += chunk; pos += len(chunk); n -= len(chunk)
        return bytes(out)

    def _abs(self, disk, offset):
        return self.starts[disk] + offset if disk < len(self.parts) else -1

    def _read_central_directory(self):
        tail_len = min(self.total, 65557 + 20)
        tail = self._read(self.total - tail_len, tail_len)
        e = tail.rfind(b"PK\x05\x06")                              # "end of central directory" record
        if e < 0:
            raise ValueError("no zip end record found: is the last part (.zip) complete?")
        this_disk, cd_disk, _, n_entries, cd_size, cd_off = struct.unpack("<HHHHII", tail[e + 4:e + 20])
        loc = tail.rfind(b"PK\x06\x07", 0, e)                       # Zip64 locator (archives > 65535 files or > 4 GB)
        if loc >= 0:
            z_disk, z_off, _ = struct.unpack("<IQI", tail[loc + 4:loc + 20])
            rec = self._read(self._abs(z_disk, z_off), 56)
            if rec[:4] != b"PK\x06\x06":
                raise ValueError("broken Zip64 record")
            this_disk, cd_disk, _, n_entries, cd_size, cd_off = struct.unpack("<IIQQQQ", rec[16:56])
        if this_disk not in (0, len(self.parts) - 1):
            raise ValueError(f"the archive says it has {this_disk + 1} parts, but {len(self.parts)} were found: a part is missing")
        cd = self._read(self._abs(cd_disk, cd_off), cd_size)
        self.members, p = [], 0
        while p + 46 <= len(cd) and cd[p:p + 4] == b"PK\x01\x02":
            (_, _, _, flags, method, _, _, crc, csize, usize, nlen, elen, clen,
             disk, _, _, off) = struct.unpack("<IHHHHHHIIIHHHHHII", cd[p:p + 46])
            raw = cd[p + 46:p + 46 + nlen]
            name = raw.decode("utf-8" if flags & 0x800 else "cp437")
            extra = cd[p + 46 + nlen:p + 46 + nlen + elen]
            q = 0
            while q + 4 <= len(extra):                             # Zip64 extra field holds the large values
                hid, hlen = struct.unpack("<HH", extra[q:q + 4])
                if hid == 1:
                    vals, r = extra[q + 4:q + 4 + hlen], 0
                    if usize == 0xFFFFFFFF: usize = struct.unpack("<Q", vals[r:r + 8])[0]; r += 8
                    if csize == 0xFFFFFFFF: csize = struct.unpack("<Q", vals[r:r + 8])[0]; r += 8
                    if off == 0xFFFFFFFF:   off = struct.unpack("<Q", vals[r:r + 8])[0]; r += 8
                    if disk == 0xFFFF:      disk = struct.unpack("<I", vals[r:r + 4])[0]
                q += 4 + hlen
            self.members.append(Member(name, usize, csize, method, crc, flags, disk, off))
            p += 46 + nlen + elen + clen
        if len(self.members) != n_entries:
            raise ValueError(f"table of contents lists {n_entries} files but {len(self.members)} were read")

    def read(self, m):
        if m.flags & 0x1:
            raise ValueError("file is encrypted")
        start = self._abs(m.disk, m.offset)
        head = self._read(start, 30)                               # local file header
        if head[:4] != b"PK\x03\x04":
            raise ValueError(f"bad local header for {m.filename}")
        nlen, elen = struct.unpack("<HH", head[26:30])
        data = self._read(start + 30 + nlen + elen, m.compress_size)
        if m.method == 8:
            data = zlib.decompressobj(-15).decompress(data)        # deflate
        elif m.method != 0:                                         # 0 = stored
            raise ValueError(f"compression method {m.method} not supported here; use the manual steps in 0.3")
        if zlib.crc32(data) & 0xFFFFFFFF != m.crc or len(data) != m.file_size:
            raise ValueError(f"checksum mismatch for {m.filename}")
        return data


def list_archives(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.is_file() and ARCHIVE_RE.search(p.name))


def group_zip_sets(arcs):
    """Group files into zip archives: name.zip (+ name.z01, name.z02 ...) or name.zip.001, .002 ...
    Returns ([(set_name, [parts in order])], [problems])."""
    sets, problems, used = [], [], set()
    for a in arcs:
        if a.suffix.lower() == ".zip":                             # split style: .z01 ... .zNN come before .zip
            zs = sorted((p for p in arcs if p.parent == a.parent and p.stem == a.stem
                         and re.fullmatch(r"\.z\d{2}", p.suffix, re.IGNORECASE)), key=lambda p: int(p.suffix[2:]))
            nums = [int(p.suffix[2:]) for p in zs]
            if nums != list(range(1, len(nums) + 1)):
                gaps = [f"{a.stem}.z{n:02d}" for n in range(1, max(nums) + 1) if n not in nums]
                problems.append(f"{a.name}: missing part(s) {gaps}; copy/shortcut them too, then rerun 0.4")
            else:
                sets.append((a.name, zs + [a]))
            used.update(zs + [a])
    numbered = defaultdict(list)                                   # byte-split style: name.zip.001, name.zip.002 ...
    for a in arcs:
        if re.fullmatch(r"\.\d{3}", a.suffix) and a.stem.lower().endswith(".zip"):
            numbered[(a.parent, a.stem)].append(a); used.add(a)
    for (_, stem), ps in numbered.items():
        sets.append((stem, sorted(ps, key=lambda p: int(p.suffix[1:]))))
    leftover = defaultdict(list)
    for a in arcs:
        if a not in used:
            leftover[a.stem].append(a.name)
    for stem, names in leftover.items():
        problems.append(f"{names[:3]}{' ...' if len(names) > 3 else ''}: not a zip archive, or its final '{stem}.zip' "
                        "part is missing; use the manual steps in 0.3")
    return sets, problems


def safe_member(name):
    p = Path(name)
    return not p.is_absolute() and ".." not in p.parts          # never write outside DATASET_DIR


def extract_sample(zf, set_name, out_dir, per_folder, seed, log, workers=8):
    """Copy a seeded random sample of images from EACH image folder inside one archive, keeping paths."""
    members = [m for m in zf.members if not m.filename.endswith("/") and safe_member(m.filename)
               and Path(m.filename).suffix.lower() in IMAGE_EXTS_S0]
    groups = defaultdict(list)
    for m in members:
        groups[str(Path(m.filename).parent)].append(m)
    for folder, ms in sorted(groups.items()):
        ms = sorted(ms, key=lambda m: m.filename)
        chosen = random.Random(f"{seed}:{set_name}:{folder}").sample(ms, min(per_folder, len(ms)))
        todo = [m for m in chosen if not ((Path(out_dir) / m.filename).exists()
                                          and (Path(out_dir) / m.filename).stat().st_size == m.file_size)]

        def copy_one(m):
            data = zf.read(m)                                      # only this file's bytes are read
            target = Path(out_dir) / m.filename
            target.parent.mkdir(parents=True, exist_ok=True)
            tmp = target.with_name(target.name + ".part")
            tmp.write_bytes(data); os.replace(tmp, target)         # never leave half-written images

        with ThreadPoolExecutor(workers) as pool:                  # several reads at once: Drive is slow per request
            list(tqdm(pool.map(copy_one, todo), total=len(todo), desc=f"{set_name} | {folder}"[-60:], leave=False))
        log.append({"archive": set_name, "generator_folder": Path(out_dir).name, "folder_in_archive": folder,
                    "images_in_folder": len(ms), "selected": len(chosen), "newly_written": len(todo)})


if DATA_SOURCE_MODE == "download_subset":
    src_root = Path(ARCHIVE_SOURCE_DIR)
    if not ARCHIVE_SOURCE_DIR or not src_root.exists():
        raise StopHere(f"ARCHIVE_SOURCE_DIR not found: '{ARCHIVE_SOURCE_DIR}'. Set it in 0.1 to the Drive folder where the "
                       "official generator folders are visible (see 0.3, step 3).")
    visible = sorted(p.name for p in src_root.iterdir() if p.is_dir())
    print("Generator folders visible in ARCHIVE_SOURCE_DIR:", visible)
    missing = [g for g in SELECTED_GENERATORS if g not in visible]
    if len(SELECTED_GENERATORS) < 3 or missing:
        raise StopHere(f"Set SELECTED_GENERATORS in 0.1 to 3 names from the list above, exactly as written. "
                       f"Selected: {SELECTED_GENERATORS}; not found: {missing}")
    per_folder = math.ceil(TARGET_PER_CLASS * 1.3)
    log, problems = [], []
    for gen in SELECTED_GENERATORS:
        arcs = list_archives(src_root / gen)
        sets, probs = group_zip_sets(arcs)
        print(f"\n{gen}: {len(arcs)} archive file(s) -> {len(sets)} zip archive(s)")
        for name, parts in sets:
            gb = sum(os.path.getsize(p) for p in parts) / 1e9
            print(f"   {name}: {len(parts)} part(s) [{parts[0].name} ... {parts[-1].name}], {gb:.1f} GB in total")
        problems += [(gen, p) for p in probs]
        if not arcs:
            problems.append((gen, "no archive files found in this folder"))
        for name, parts in sets:
            try:
                t0 = time.time()
                zf = SplitZip(parts)
                print(f"   {name}: table of contents read ({len(zf.members):,} entries, {time.time() - t0:.0f}s)")
                extract_sample(zf, name, Path(DATASET_DIR) / gen, per_folder, SEED_S0, log)
            except Exception as e:
                problems.append((gen, f"{name}: {type(e).__name__}: {e}"))
    setup_dir = Path(DATASET_DIR) / SETUP_DIRNAME
    setup_dir.mkdir(parents=True, exist_ok=True)
    EXTRACTION_RECORD = {"time": datetime.datetime.now().isoformat(timespec="seconds"), "archive_source_dir": ARCHIVE_SOURCE_DIR,
                         "per_folder": per_folder, "seed": SEED_S0, "entries": log, "problems": problems}
    (setup_dir / "subset_extraction_log.json").write_text(json.dumps(EXTRACTION_RECORD, indent=2))
    if log:
        print("\nExtracted (per image folder inside each archive):")
        print(pd.DataFrame(log)[["generator_folder", "folder_in_archive", "images_in_folder", "selected",
                                 "newly_written"]].to_string(index=False))
    for gen, msg in problems:
        print(f"PROBLEM  {gen}: {msg}")
    done = {e["generator_folder"] for e in log}
    not_done = [g for g in SELECTED_GENERATORS if g not in done]
    if not_done:
        raise StopHere(f"No images could be extracted automatically for: {not_done}. Use the manual steps in 0.3 for "
                       "these generators, then run Section 0 again with DATA_SOURCE_MODE = 'existing_drive'.")
    if problems:
        print("\nSome archives had problems (listed above). Check them before trusting the counts in 0.6.")
else:
    print(f"Mode is '{DATA_SOURCE_MODE}': no extraction.")

### 0.5 · Inspect `DATASET_DIR`: which generator folders exist?
Lists the folders **directly inside** `DATASET_DIR`. Each one is a candidate generator, keeping its exact name. It also shows a shallow tree, which lists folder names only, so it stays fast even on huge folders.

If you see a single wrapper folder (such as `GenImage/`) and the generators are *inside* it, change `DATASET_DIR` in 0.1 to point one level deeper.

In [ ]:
ROOT_S0 = Path(DATASET_DIR)
if not ROOT_S0.exists():
    raise StopHere(f"DATASET_DIR does not exist: {DATASET_DIR}\n"
                   "Workflow A: set DATASET_DIR (0.1) to your extracted GenImage folder (copy the path from Colab's Files panel).\n"
                   "Workflow B: follow 0.3 first.")
TOP_FOLDERS = sorted(p.name for p in ROOT_S0.iterdir() if p.is_dir() and not p.name.startswith(".") and p.name != SETUP_DIRNAME)
top_archives = sorted(p.name for p in ROOT_S0.iterdir() if p.is_file() and ARCHIVE_RE.search(p.name))


def shallow_tree(path, depth=0, max_depth=3):
    """Folder names only, 3 levels deep (the image folders themselves are not listed, so this is fast)."""
    if depth >= max_depth:
        return
    subs = sorted(p for p in path.iterdir() if p.is_dir() and not p.name.startswith(".") and p.name != SETUP_DIRNAME)
    for p in subs:
        print("    " * depth + f"{p.name}/")
        if depth + 1 < max_depth:
            shallow_tree(p, depth + 1, max_depth)


print(f"DATASET_DIR: {DATASET_DIR}\n")
shallow_tree(ROOT_S0)
print(f"\nCandidate generator folders ({len(TOP_FOLDERS)}):")
for name in TOP_FOLDERS:
    print(f"   {name!r}")
if top_archives:
    print("\nArchive files directly inside DATASET_DIR (not extracted?):", top_archives)
if not SELECTED_GENERATORS:
    raise StopHere("Now set SELECTED_GENERATORS in 0.1 to 3 of the names above (copy them exactly, with quotes), then rerun Section 0.")

### 0.6 · Count images in the selected generators, and suggest a mapping
For each selected generator, every image folder is counted and any leftover archives are reported. Unselected generators are **not** scanned, which saves time.

The cell then prints a **suggested mapping**: which folder holds which generator and label. Labels are guessed **only** from folder names (`ai`, `fake`, `generated` → AI; `nature`, `real` → Real), so check every line. The generator name is always the top-level folder name, unchanged.

In [ ]:
missing = [g for g in SELECTED_GENERATORS if g not in TOP_FOLDERS]
if missing or len(SELECTED_GENERATORS) < 3:
    raise StopHere(f"Selected generators must be at least 3 names from {TOP_FOLDERS}.\n"
                   f"Selected: {SELECTED_GENERATORS}\nNot found in DATASET_DIR: {missing}")
if len(SELECTED_GENERATORS) > 3:
    print(f"NOTE: {len(SELECTED_GENERATORS)} generators selected. M1 holds out one and trains on all the others.")

FILES_S0, ARCHIVES_FOUND, rows = {}, {}, []
for gen in SELECTED_GENERATORS:
    t0 = time.time()
    for dirpath, dirnames, filenames in os.walk(ROOT_S0 / gen):
        dirnames.sort()
        rel = Path(dirpath).relative_to(ROOT_S0).as_posix()
        imgs = sorted(f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTS_S0 and not f.startswith("."))
        arcs = [f for f in filenames if ARCHIVE_RE.search(f)]
        if arcs:
            ARCHIVES_FOUND.setdefault(gen, []).extend(f"{rel}/{a}" for a in arcs)
        if imgs:
            FILES_S0[rel] = imgs
            rows.append({"generator": gen, "folder": rel, "images": len(imgs),
                         "extensions": dict(Counter(Path(f).suffix.lower() for f in imgs))})
    print(f"{gen}: scanned in {time.time() - t0:.0f}s")
COUNTS_S0 = pd.DataFrame(rows)
if COUNTS_S0.empty:
    raise StopHere("No images found inside the selected generator folders. Are the archives extracted? See 0.3.")
print("\nImage folders found:")
print(COUNTS_S0.to_string(index=False))
for gen, arcs in ARCHIVES_FOUND.items():
    print(f"\nArchive files still inside '{gen}' ({len(arcs)}): {arcs[:5]}{' ...' if len(arcs) > 5 else ''}")

AI_TOKENS, REAL_TOKENS = {"ai", "fake", "fakes", "generated", "synthetic"}, {"nature", "real", "natural", "reals"}


def label_from_name(part):
    tokens = set(re.split(r"[^a-z0-9]+", part.lower()))
    return 0 if tokens & REAL_TOKENS else (1 if tokens & AI_TOKENS else None)


def suggest_rule(rel):
    """Rule folder = the path up to the DEEPEST folder whose name says ai/nature; label from that name."""
    parts = rel.split("/")
    for i in range(len(parts) - 1, 0, -1):
        lab = label_from_name(parts[i])
        if lab is not None:
            return "/".join(parts[:i + 1]), lab
    return rel, None


SUGGESTED_RULES = {}
for _, r in COUNTS_S0.iterrows():
    folder, lab = suggest_rule(r["folder"])
    key = (folder, r["generator"])
    if key not in SUGGESTED_RULES:
        SUGGESTED_RULES[key] = {"folder": folder, "generator": r["generator"], "label": lab, "images": 0}
    SUGGESTED_RULES[key]["images"] += int(r["images"])
SUGGESTED_RULES = list(SUGGESTED_RULES.values())

print("\nSuggested mapping. CHECK every label (1 = AI, 0 = Real). Paste into 0.7 only if you need to change it:\n")
print("FOLDER_RULES_S0 = [")
for r in SUGGESTED_RULES:
    lab = r["label"] if r["label"] is not None else '"CHANGE_ME"'
    note = "AI" if r["label"] == 1 else ("Real" if r["label"] == 0 else "label unknown: set 0 or 1")
    print(f'    {{"folder": {r["folder"]!r}, "generator": {r["generator"]!r}, "label": {lab}}},   # {r["images"]:,} images, {note}')
print("]")

### 0.7 · Confirm the folder → generator / label mapping ✏️
- **If the suggestion in 0.6 is correct**, leave `FOLDER_RULES_S0 = []` and the suggestion is used.
- **If anything is wrong** (a label, or a folder that should be left out), paste the printed list into `FOLDER_RULES_S0` and fix it.
- Put folders you deliberately want to leave out in `EXTRA_IGNORE_S0`. For example, if you only want each generator's `train` part.

You give final confirmation in **0.11**, after seeing sample images in 0.9.

In [ ]:
FOLDER_RULES_S0 = []     # <<< CHANGE only if the suggestion in 0.6 is wrong (paste it here and edit)
EXTRA_IGNORE_S0 = []     # <<< optional: folders (relative to DATASET_DIR) to leave out on purpose

if not FOLDER_RULES_S0:
    unknown = [r for r in SUGGESTED_RULES if r["label"] not in (0, 1)]
    if unknown:
        raise StopHere("Some labels could not be guessed from folder names. Paste the list from 0.6 into "
                       f"FOLDER_RULES_S0 and set each label to 0 (Real) or 1 (AI): {[r['folder'] for r in unknown]}")
    FOLDER_RULES_S0 = [{k: r[k] for k in ("folder", "generator", "label")} for r in SUGGESTED_RULES]
    print("Using the suggested mapping from 0.6.")

for r in FOLDER_RULES_S0:
    assert r["label"] in (0, 1), f"Label must be 0 or 1: {r}"
    assert r["generator"] in SELECTED_GENERATORS, f"Generator must be one of {SELECTED_GENERATORS}: {r}"
    assert r["folder"] == r["generator"] or r["folder"].startswith(r["generator"] + "/"), \
        f"A rule's folder must be inside its own generator folder (generators are never mixed): {r}"


def _matches(rel, folder):
    return rel == folder or rel.startswith(folder.rstrip("/") + "/")


rows, UNMAPPED_S0 = [], []
for _, r in COUNTS_S0.iterrows():
    if any(_matches(r["folder"], ig) for ig in EXTRA_IGNORE_S0):
        continue
    hits = [x for x in FOLDER_RULES_S0 if _matches(r["folder"], x["folder"])]
    if not hits:
        UNMAPPED_S0.append(r["folder"])
        continue
    rule = max(hits, key=lambda x: len(x["folder"]))
    rows.append({"generator": rule["generator"], "label": rule["label"], "rule_folder": rule["folder"],
                 "folder": r["folder"], "images": int(r["images"])})
MAPPED_S0 = pd.DataFrame(rows)
CLASS_COUNTS_S0 = (MAPPED_S0.pivot_table(index="generator", columns="label", values="images", aggfunc="sum", fill_value=0)
                   .reindex(columns=[0, 1], fill_value=0).rename(columns={0: "real (0)", 1: "ai (1)"}))
print("Images per generator and class under this mapping:")
print(CLASS_COUNTS_S0)
if UNMAPPED_S0:
    print("\nWARNING: image folders with NO rule (add a rule or list them in EXTRA_IGNORE_S0):", UNMAPPED_S0)

### 0.8 · Formats, image sizes and corrupted files
For each mapped folder, up to `CHECK_PER_FOLDER` images are opened **fully**, which detects truncated or broken files, not just bad headers. The cell records the real file format, size and colour mode. It reads in parallel because Drive is slow. A few broken files are tolerated, since the M1 pipeline excludes unreadable files later. A folder with no readable images fails the check in 0.11.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image


def check_image(path):
    try:
        with Image.open(path) as im:
            im.verify()                                    # checks the file structure
        with Image.open(path) as im:
            im.load()                                      # decodes every pixel: catches truncated files
            return {"ok": True, "format": im.format, "width": im.size[0], "height": im.size[1], "mode": im.mode}
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"[:150]}


rows, CORRUPT_S0 = [], []
for _, m in MAPPED_S0.iterrows():
    names = FILES_S0[m["folder"]]
    pick = names if CHECK_PER_FOLDER is None else random.Random(f"{SEED_S0}:{m['folder']}").sample(names, min(CHECK_PER_FOLDER, len(names)))
    paths = [ROOT_S0 / m["folder"] / f for f in pick]
    with ThreadPoolExecutor(max_workers=16) as ex:
        res = list(tqdm(ex.map(check_image, paths), total=len(paths), desc=m["folder"][-50:], leave=False))
    good = [r for r in res if r["ok"]]
    CORRUPT_S0 += [{"file": str(p), "error": r["error"]} for p, r in zip(paths, res) if not r["ok"]]
    w = [r["width"] for r in good]; h = [r["height"] for r in good]
    rows.append({"generator": m["generator"], "label": m["label"], "folder": m["folder"], "images": m["images"],
                 "checked": len(res), "corrupt": len(res) - len(good),
                 "formats": dict(Counter(r["format"] for r in good)), "modes": dict(Counter(r["mode"] for r in good)),
                 "width_median": float(pd.Series(w).median()) if w else None,
                 "height_median": float(pd.Series(h).median()) if h else None,
                 "size_range": f"{min(w)}x{min(h)} .. {max(w)}x{max(h)}" if w else ""})
QUALITY_S0 = pd.DataFrame(rows)
print(QUALITY_S0.drop(columns=["folder"]).to_string(index=False))
print(f"\nCorrupted / unreadable files found: {len(CORRUPT_S0)}")
for c in CORRUPT_S0[:10]:
    print("  ", c["file"], "->", c["error"])

### 0.9 · Look at sample Real and AI images
Three Real and three AI images per selected generator. **This is your label check.** Do the AI images look generated? Do the Real ones look like ordinary photos? If not, fix the mapping in 0.7.

In [ ]:
import matplotlib.pyplot as plt

rng = random.Random(SEED_S0)
fig, axes = plt.subplots(len(SELECTED_GENERATORS), 6, figsize=(14, 2.7 * len(SELECTED_GENERATORS)), squeeze=False)
for i, gen in enumerate(SELECTED_GENERATORS):
    j = 0
    for lab in (0, 1):
        folders = MAPPED_S0[(MAPPED_S0.generator == gen) & (MAPPED_S0.label == lab)]["folder"].tolist()
        pool = [(f, n) for f in folders for n in FILES_S0[f][:2000]]
        for f, n in rng.sample(pool, min(3, len(pool))):
            with Image.open(ROOT_S0 / f / n) as im:
                axes[i][j].imshow(im.convert("RGB"))
                axes[i][j].set_title(f"{'AI' if lab else 'Real'} | {im.size[0]}x{im.size[1]} {im.format}",
                                     fontsize=8, color="#1f5fbf" if lab else "#2b8a3e")
            j += 1
    for ax in axes[i]:
        ax.set_xticks([]); ax.set_yticks([])
    axes[i][0].set_ylabel(gen[:22], fontsize=9)
plt.suptitle("Sample images per generator: Real (left) and AI (right)")
plt.tight_layout()
(ROOT_S0 / SETUP_DIRNAME).mkdir(exist_ok=True)
plt.savefig(ROOT_S0 / SETUP_DIRNAME / "dataset_samples.png", dpi=110)
plt.show()

### 0.10 · Dataset summary CSV and DATASET_README.md
Writes, inside `DATASET_DIR`:
- `DATASET_SUMMARY.csv`: one row per mapped folder, with counts, formats, sizes and corrupt counts, plus totals per generator and class
- `DATASET_README.md`: source, selected generators, image counts, label mapping, folder mapping, download date and subset level

Everything in these files is measured or confirmed by you. Nothing is filled in by guesswork.

In [ ]:
summary = QUALITY_S0.copy()
summary.insert(0, "class", summary["label"].map({0: "real", 1: "ai"}))
totals = (summary.groupby(["generator", "class", "label"], as_index=False)[["images", "checked", "corrupt"]].sum()
          .assign(folder="ALL FOLDERS"))
SUMMARY_S0 = pd.concat([summary, totals], ignore_index=True)
SUMMARY_S0.to_csv(ROOT_S0 / "DATASET_SUMMARY.csv", index=False)

extraction = (ROOT_S0 / SETUP_DIRNAME / "subset_extraction_log.json")
extracted_at = json.loads(extraction.read_text())["time"] if extraction.exists() else None
download_date = DOWNLOAD_DATE or "not recorded (set DOWNLOAD_DATE in 0.1)"
lines = [
    "# GenImage subset used by VeriStamp", "",
    f"- **Source:** official GenImage dataset, {GENIMAGE_SOURCE['repository']}",
    f"  - Paper: {GENIMAGE_SOURCE['paper']}",
    f"  - Google Drive: {GENIMAGE_SOURCE['google_drive']}",
    f"  - Licence: {GENIMAGE_SOURCE['license']}",
    f"- **How it was obtained:** DATA_SOURCE_MODE = `{DATA_SOURCE_MODE}`",
    f"- **Date of download:** {download_date}",
] + ([f"- **Subset extracted by this notebook on:** {extracted_at} (log: `{SETUP_DIRNAME}/subset_extraction_log.json`)"] if extracted_at else []) + [
    f"- **Subset level:** {SUBSET_LEVEL} ({TARGET_PER_CLASS} images per class per generator will be used by the M1 split)",
    f"- **Summary written:** {datetime.datetime.now().isoformat(timespec='seconds')}", "",
    "## Selected generators", "", *[f"- `{g}`" for g in SELECTED_GENERATORS], "",
    "## Label mapping", "", "- Real (nature) = **0**", "- AI-generated = **1**", "",
    "## Image counts (all files found under the mapping)", "",
    "| generator | real (0) | ai (1) |", "|---|---|---|",
    *[f"| {g} | {int(r['real (0)']):,} | {int(r['ai (1)']):,} |" for g, r in CLASS_COUNTS_S0.iterrows()], "",
    "## Folder mapping (folder relative to DATASET_DIR -> generator, label)", "",
    "| folder | generator | label |", "|---|---|---|",
    *[f"| `{r['folder']}` | {r['generator']} | {r['label']} ({'AI' if r['label'] else 'Real'}) |" for r in FOLDER_RULES_S0], "",
    f"Folders left out on purpose: {EXTRA_IGNORE_S0 or 'none'}. Generators present but not selected: "
    f"{[t for t in TOP_FOLDERS if t not in SELECTED_GENERATORS] or 'none'}.", "",
    "## Quality check", "",
    f"- Images opened per folder: {CHECK_PER_FOLDER or 'all'}; corrupted/unreadable found: {len(CORRUPT_S0)}",
    f"- Formats and sizes per folder: see `DATASET_SUMMARY.csv`", "",
    "Train / validation / test splits are NOT defined here. They are created by the M1 notebook (Section 7) "
    "and saved under the project folder, so that every model reuses the same split.",
]
(ROOT_S0 / "DATASET_README.md").write_text("\n".join(lines))
print(SUMMARY_S0[SUMMARY_S0["folder"] == "ALL FOLDERS"][["generator", "class", "images", "checked", "corrupt"]].to_string(index=False))
print("\nWrote", ROOT_S0 / "DATASET_SUMMARY.csv", "and", ROOT_S0 / "DATASET_README.md")

### 0.11 · Verify, then hand over to Section 1 ✏️
Set `LABELS_CONFIRMED = True` **only after** you have checked the mapping (0.7) and the sample images (0.9).

All of these must pass:

| Check | Why |
|---|---|
| DATA_ROOT exists | the pipeline reads from it |
| at least 3 selected generators present | leave-one-generator-out needs 2 seen + 1 unseen |
| each generator has AI **and** Real images | otherwise the split can't be balanced |
| each folder has readable images | broken copies would silently shrink the data |
| enough images for the chosen subset level | `N_PER_CLASS` must be reachable after duplicates are removed |
| labels confirmed by you | labels are never trusted without a human check |
| no archives left inside selected generators | unless you set `ALLOW_ARCHIVES_IN_SELECTED = True` |
| every image folder is mapped or left out on purpose | nothing is skipped silently |

If everything passes, the cell saves `DATASET_SETUP.json` in `DATASET_DIR`, with the selected generators, the confirmed mapping and the counts. It then prints the exact `DATA_ROOT` and `N_PER_CLASS` lines for Section 1. Section 4 refuses to run unless this file says `verified: true`.

In [ ]:
LABELS_CONFIRMED = False    # <<< CHANGE to True after checking the mapping (0.7) and the sample images (0.9)

checks = []
def add(name, ok, detail="", level="FAIL"):
    checks.append({"check": name, "result": "PASS" if ok else level, "detail": detail})

add("DATA_ROOT exists", ROOT_S0.exists(), DATASET_DIR)
present = [g for g in SELECTED_GENERATORS if g in TOP_FOLDERS]
add("At least 3 selected generators are present", len(present) >= 3,
    f"present: {present}; missing: {[g for g in SELECTED_GENERATORS if g not in TOP_FOLDERS]}")
need, rec = TARGET_PER_CLASS, math.ceil(TARGET_PER_CLASS * 1.3)
for gen in SELECTED_GENERATORS:
    n_real = int(CLASS_COUNTS_S0.loc[gen, "real (0)"]) if gen in CLASS_COUNTS_S0.index else 0
    n_ai = int(CLASS_COUNTS_S0.loc[gen, "ai (1)"]) if gen in CLASS_COUNTS_S0.index else 0
    add(f"'{gen}' has AI images", n_ai > 0, f"{n_ai:,}")
    add(f"'{gen}' has Real images", n_real > 0, f"{n_real:,}")
    add(f"'{gen}' has >= {need} per class ({SUBSET_LEVEL})", min(n_ai, n_real) >= need, f"real {n_real:,} / ai {n_ai:,}")
    if need <= min(n_ai, n_real) < rec:
        add(f"'{gen}' has the recommended {rec} per class (30% spare for duplicates)", False,
            f"real {n_real:,} / ai {n_ai:,}", level="WARNING")
no_readable = QUALITY_S0[QUALITY_S0["checked"] == QUALITY_S0["corrupt"]]["folder"].tolist()
add("Every mapped folder has readable images", not no_readable, f"folders with none readable: {no_readable}")
if CORRUPT_S0:
    add("No corrupted files in the checked sample", False, f"{len(CORRUPT_S0)} corrupted (M1 will exclude them)", level="WARNING")
add("Labels confirmed by you (LABELS_CONFIRMED = True)", LABELS_CONFIRMED)
add("No unextracted archives inside the selected generators", not ARCHIVES_FOUND or ALLOW_ARCHIVES_IN_SELECTED,
    f"{sum(len(v) for v in ARCHIVES_FOUND.values())} archive file(s)" + (" (allowed by you)" if ALLOW_ARCHIVES_IN_SELECTED else ""))
add("Every image folder is mapped or left out on purpose", not UNMAPPED_S0, f"unmapped: {UNMAPPED_S0}")

VERIFY_S0 = pd.DataFrame(checks)
print(VERIFY_S0.to_string(index=False))
if (VERIFY_S0["result"] == "FAIL").any():
    raise StopHere("Dataset setup is NOT verified. Fix the FAIL rows above and rerun Section 0. Do not continue to Section 1.")

ignore = sorted(set([t for t in TOP_FOLDERS if t not in SELECTED_GENERATORS] + EXTRA_IGNORE_S0 + [SETUP_DIRNAME]))
max_n = int(CLASS_COUNTS_S0.loc[SELECTED_GENERATORS].min().min())
DATASET_SETUP = {
    "verified": True, "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "data_root": DATASET_DIR, "data_source_mode": DATA_SOURCE_MODE, "source": GENIMAGE_SOURCE,
    "download_date": DOWNLOAD_DATE or None, "subset_level": SUBSET_LEVEL, "target_per_class": TARGET_PER_CLASS,
    "selected_generators": SELECTED_GENERATORS, "folder_rules": FOLDER_RULES_S0, "ignore_folders": ignore,
    "label_mapping": {"real": 0, "ai": 1},
    "class_counts": {g: {"real": int(r["real (0)"]), "ai": int(r["ai (1)"])} for g, r in CLASS_COUNTS_S0.iterrows()},
    "max_n_per_class": max_n, "corrupt_in_sample": len(CORRUPT_S0),
}
(ROOT_S0 / "DATASET_SETUP.json").write_text(json.dumps(DATASET_SETUP, indent=2))
print("\nDATASET SETUP VERIFIED. Saved", ROOT_S0 / "DATASET_SETUP.json")
print("\nCopy these two lines into Section 1:\n")
print(f'DATA_ROOT   = "{DATASET_DIR}"')
print(f"N_PER_CLASS = {TARGET_PER_CLASS}")
print(f"\n(The largest possible N_PER_CLASS with this data is {max_n:,}. Section 7 will ask for UNSEEN_GENERATOR, "
      f"one of: {SELECTED_GENERATORS})")

**Expected output of Section 0:**
- 0.5: the list of generator folders
- 0.6: image folders with counts, and the suggested mapping
- 0.7: a Real/AI count per generator
- 0.8: a quality table
- 0.9: a picture grid
- 0.10: the two files written
- 0.11: a table where every row says PASS, then the `DATA_ROOT` and `N_PER_CLASS` lines

📤 **Send me:** the outputs of 0.5, 0.6, 0.7, 0.8 and 0.11, plus a screenshot of 0.9.

## 1 · Project configuration
**Purpose:** put every setting in one place. The whole `CONFIG` dictionary is saved as `config.yaml` with each experiment, so any result can be traced back to exactly how it was made.

✏️ **Edit only the lines marked `<<< CHANGE`.** `DATA_ROOT` and `N_PER_CLASS` come from the lines printed at the end of Section 0. `UNSEEN_GENERATOR` is set in Section 7.

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════╗
# ║   VALUES YOU MUST CHECK OR CHANGE  (marked  <<< CHANGE)                 ║
# ╚════════════════════════════════════════════════════════════════════════╝
DATA_ROOT   = "/content/drive/MyDrive/CHANGE_ME"   # <<< CHANGE: copy the DATA_ROOT line printed at the end of Section 0
PROJECT_DIR = "/content/drive/MyDrive/VeriStamp"    # <<< CHANGE (optional): where splits, results and the code library are saved
RUN_NAME    = "m1_run_001"                          # <<< CHANGE for every new experiment (never reuse a name for new settings)
N_PER_CLASS = 200    # <<< CHANGE: copy from Section 0 (200 = debug, 500 = interim, 3000-5000 = final)
EPOCHS      = 10     # <<< CHANGE if needed: maximum epochs (early stopping may stop sooner)
RESUME      = True   # True = continue an interrupted run from last.pt instead of starting again
# ══════════════════════════════════════════════════════════════════════════

SEED = 42            # one fixed seed for sampling, splitting, JPEG qualities and training

CONFIG = {
    "project": {"name": "VeriStamp", "component": "M1", "run_name": RUN_NAME},
    "seed": SEED,
    "paths": {
        "data_root": DATA_ROOT,
        "project_dir": PROJECT_DIR,
        "splits_dir": f"{PROJECT_DIR}/data/splits",          # saved splits, reused by M2 and Fusion
        "manifests_dir": f"{PROJECT_DIR}/data/manifests",
        "experiments_dir": f"{PROJECT_DIR}/experiments",
        "cache_local": "/content/cache",                      # fast Colab disk (wiped when the session ends)
        "cache_drive": f"{PROJECT_DIR}/cache",                # copy on Drive, so later sessions skip preprocessing
        "lib_drive": f"{PROJECT_DIR}/vslib",                  # the shared code library
    },
    "data": {
        "dataset": "genimage",
        "selected_generators": None,     # filled from Section 0 (DATASET_SETUP.json) in Section 4
        "dataset_subset_level": None,    # filled from Section 0 in Section 4
        "folder_rules": None,            # set in Section 5 (taken from Section 0 unless you override it)
        "ignore_folders": [],            # set in Section 5
        "unseen_generator": None,        # set in Section 7
        "n_per_class": N_PER_CLASS,
        "fractions": {"train": 0.70, "val": 0.15, "test_seen": 0.15},   # for the SEEN generators
        "oversample": 1.3,               # hash 30% extra candidates to replace duplicates/unreadable files
        "drop_near_duplicates": True,    # exclude near-identical copies that land in different splits
        "hash_workers": 16,
    },
    "preprocess": {                      # identical for Real and AI images (proposal bias control)
        "resize_short_side": 256,
        "cache_size": 256,
        "crop_size": 224,
        "jpeg_quality_range": [75, 95],
        "train_hflip": True,             # augmentation, training only
        "train_rescale_range": None,     # e.g. [0.5, 2.0] = random rescale, training only (None = off, as in m1_run_001)
    },
    "features": {"hp_kernel": 5, "hp_sigma": 1.0, "stats_batches": 20},
    "model": {"arch": "resnet50", "pretrained": True, "weights": "IMAGENET1K_V2", "dropout": 0.3},
    "train": {
        "epochs": EPOCHS, "batch_size": 64, "cpu_batch_size": 8,
        "lr_backbone": 1e-4, "lr_new": 1e-3, "weight_decay": 1e-4,
        "warmup_epochs": 1, "early_stop_patience": 3, "num_workers": 2,
        "max_train_batches": None, "max_val_batches": None,      # None = use everything
    },
    "sanity": {"overfit_images": 32, "overfit_steps_gpu": 80, "overfit_steps_cpu": 30,
               "shuffled_train_images": 512, "shuffled_epochs": 2},
    "runs": {"modes": ["rgb", "hp_fft"]},   # "rgb" = baseline, "hp_fft" = proposal M1, ("rgb_hp_fft" optional)
}
assert CONFIG["preprocess"]["resize_short_side"] >= CONFIG["preprocess"]["cache_size"] >= CONFIG["preprocess"]["crop_size"]
print(f"Run name: {RUN_NAME}   images per class per generator: {N_PER_CLASS}   max epochs: {EPOCHS}")
print("Models to train:", CONFIG["runs"]["modes"])

**Expected output:** one line with your run name, `N_PER_CLASS` and epochs, then `Models to train: ['rgb', 'hp_fft']`.

## 2 · Install / import packages, and write the shared code library
**Purpose:** Colab already includes PyTorch, torchvision, scikit-learn, pandas, matplotlib, Pillow, PyYAML and tqdm, so nothing needs installing.

This section:
1. connects Google Drive
2. writes the reusable modules into `vslib/`, one cell per module, each starting with `%%writefile`
3. imports them and saves a copy to Drive, where M2, Fusion and M3 notebooks will import the same code

Each module has comments. The important functions are explained again, with pictures, in the section where they are used.

In [ ]:
import os, sys, shutil, importlib, json, math, time, random
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab: skipping Google Drive mount.")

os.makedirs("vslib", exist_ok=True)
print("Writing the library to:", os.path.abspath("vslib"))

**2.1 · `vs_common.py`**: shared utilities for EVERY VeriStamp model (M1, M2, Fusion, M3).

In [ ]:
%%writefile vslib/vs_common.py
"""vs_common: shared utilities for EVERY VeriStamp model (M1, M2, Fusion, M3).

Seeds, device + mixed-precision choice, run folders, checkpoints,
JSON/YAML/CSV saving, environment info and the experiment log.
Nothing here is specific to one model or one dataset.
"""
from __future__ import annotations

import datetime
import inspect
import json
import math
import os
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml


def now() -> str:
    return datetime.datetime.now().isoformat(timespec="seconds")


def set_seed(seed: int) -> None:
    """Make Python, NumPy and PyTorch random numbers repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # safe to call without a GPU
    os.environ["PYTHONHASHSEED"] = str(seed)


def get_device() -> torch.device:
    """CUDA GPU (Colab) -> Apple GPU (Mac) -> CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def amp_settings(device: torch.device) -> dict:
    """Mixed precision: bfloat16 on A100-class GPUs, float16 + GradScaler on older GPUs, off on CPU."""
    if device.type == "cuda":
        if torch.cuda.is_bf16_supported():
            return {"enabled": True, "dtype": torch.bfloat16, "use_scaler": False}
        return {"enabled": True, "dtype": torch.float16, "use_scaler": True}
    return {"enabled": False, "dtype": torch.float32, "use_scaler": False}


def clean(obj):
    """Turn Paths, NumPy numbers, dtypes and NaN into plain JSON/YAML values."""
    if isinstance(obj, dict):
        return {str(k): clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean(v) for v in obj]
    if isinstance(obj, (Path, str)):
        return str(obj)                    # also turns str subclasses (e.g. torch.__version__) into plain str
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        v = float(obj)
        return None if (math.isnan(v) or math.isinf(v)) else v
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, torch.Tensor):
        return clean(obj.tolist())
    return obj


def save_json(obj, path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(clean(obj), indent=2))


def load_json(path):
    return json.loads(Path(path).read_text())


def save_yaml(obj, path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        yaml.safe_dump(clean(obj), f, sort_keys=False)


def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)


def environment_info() -> dict:
    info = {"timestamp": now(), "python": platform.python_version(), "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(), "device": str(get_device())}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["gpu_memory_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    return info


def make_run_dir(path, resume: bool = True) -> Path:
    """Create <run>/ and <run>/plots/. Refuses to reuse a non-empty folder unless resume=True."""
    path = Path(path)
    if path.exists() and any(path.iterdir()) and not resume:
        raise FileExistsError(f"{path} already exists. Choose a new RUN_NAME, or set RESUME = True to continue it.")
    (path / "plots").mkdir(parents=True, exist_ok=True)
    return path


def save_checkpoint(path, state: dict) -> None:
    """Save to a temporary file, then rename: a crash mid-save cannot corrupt the previous checkpoint."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)


def load_checkpoint(path, device="cpu") -> dict:
    return torch.load(path, map_location=device, weights_only=False)


def append_csv_row(path, row: dict) -> None:
    """Append one row to a CSV log (creates it, and adds new columns when needed)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame([clean(row)])
    if path.exists() and path.stat().st_size > 0:
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)


def df_to_markdown(df, floatfmt: str = ".4f") -> str:
    """A pandas table as a Markdown table (no extra packages needed)."""
    cols = list(df.columns)
    fmt = lambda v: format(v, floatfmt) if isinstance(v, (float, np.floating)) else str(v)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(fmt(r[c]) for c in cols) + " |" for _, r in df.iterrows()]
    return "\n".join(lines)


def show_source(fn) -> None:
    """Print a function's code (used to explain library functions inside the notebook)."""
    print(inspect.getsource(fn))

**2.2 · `vs_data.py`**: generic image-folder utilities (scanning, hashing, duplicate detection).

In [ ]:
%%writefile vslib/vs_data.py
"""vs_data: generic image-folder utilities (scanning, hashing, duplicate detection).

Dataset-agnostic on purpose: used for GenImage now (M1, M2, Fusion) and for the
separate COCO manifest that M3 will build later.
"""
from __future__ import annotations

import hashlib
import io
import os
import random
import statistics
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}


def _skipped(rel, skip):
    return any(rel == s or rel.startswith(s.rstrip("/") + "/") for s in skip)


def scan_image_folders(root, skip=()):
    """Walk `root` once. Returns ({relative folder: [image file names]}, Counter of non-image extensions).
    Folders listed in `skip` (relative to root) are not entered at all, e.g. generators you did not select."""
    root = Path(root)
    folders, others = {}, Counter()
    for dirpath, dirnames, filenames in os.walk(root):
        rel = Path(dirpath).relative_to(root).as_posix()  # "." means DATA_ROOT itself
        dirnames[:] = sorted(d for d in dirnames if not _skipped(d if rel == "." else f"{rel}/{d}", skip))
        imgs = sorted(f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTS and not f.startswith("."))
        if imgs:
            folders[rel] = imgs
        for f in filenames:
            ext = Path(f).suffix.lower()
            if ext not in IMAGE_EXTS:
                others[ext or "(no extension)"] += 1
    return folders, others


def print_tree(folders, root_label, max_depth=4, max_lines=150):
    """Folder tree; each number includes all sub-folders."""
    below = Counter()
    for rel, files in folders.items():
        parts = () if rel == "." else tuple(rel.split("/"))
        for d in range(min(len(parts), max_depth) + 1):
            below[parts[:d]] += len(files)
    for i, parts in enumerate(sorted(below)):
        if i >= max_lines:
            print(f"... ({len(below) - max_lines} more lines; raise max_lines to see them)")
            break
        name = parts[-1] if parts else root_label
        print("    " * len(parts) + f"{name}/  {below[parts]:,} images")


def folder_stats(root, folders, per_folder=20, max_folders=40, seed=0) -> pd.DataFrame:
    """Sample a few images per folder: size, file format, colour mode (reads headers only)."""
    rng, root, rows = random.Random(seed), Path(root), []
    for rel, files in list(folders.items())[:max_folders]:
        w, h, modes, fmts, broken = [], [], Counter(), Counter(), 0
        for f in rng.sample(files, min(per_folder, len(files))):
            try:
                with Image.open(root / rel / f) as im:
                    w.append(im.size[0]); h.append(im.size[1])
                    modes[im.mode] += 1; fmts[im.format] += 1
            except Exception:
                broken += 1
        rows.append({"folder": rel, "images": len(files),
                     "width_median": statistics.median(w) if w else None,
                     "width_range": f"{min(w)}-{max(w)}" if w else "",
                     "height_median": statistics.median(h) if h else None,
                     "height_range": f"{min(h)}-{max(h)}" if h else "",
                     "formats": dict(fmts), "modes": dict(modes), "unreadable": broken})
    return pd.DataFrame(rows)


def shared_name_report(folders, top=10):
    """How many file NAMES occur in more than one folder (a hint of duplicate photos)."""
    name_to = defaultdict(set)
    for rel, files in folders.items():
        for f in files:
            name_to[f].add(rel)
    shared = [fs for fs in name_to.values() if len(fs) > 1]
    pairs = Counter()
    for fs in shared[:50_000]:
        fs = sorted(fs)
        for i in range(len(fs)):
            for j in range(i + 1, len(fs)):
                pairs[(fs[i], fs[j])] += 1
    return len(shared), pairs.most_common(top)


def _dhash(img, n: int = 16) -> str:
    """256-bit 'difference hash' (64 hex characters): does each pixel of a 17x16 grey thumbnail get
    brighter or darker to the right? Re-saved or resized copies of one picture differ in only a few
    bits; different pictures differ in many (tested: copies <= 10 bits, different photos >= 47)."""
    g = np.asarray(img.convert("L").resize((n + 1, n), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return f"{int(''.join('1' if b else '0' for b in bits), 2):0{n * n // 4}x}"


def near_duplicate_groups(df, max_distance: int = 12, hash_col: str = "dhash") -> pd.Series:
    """Group near-identical images: dHash Hamming distance <= max_distance.

    Returns a Series (same index as df) with a group number for images that have at least one
    near-copy, and NaN otherwise. Fast on large sets: the 256 bits are cut into 16 bands; two
    hashes within 12 bits of each other must share at least one band exactly, so only images
    sharing a band are compared.
    """
    h = df[hash_col].dropna()
    ids, hashes = list(h.index), [int(x, 16) for x in h]
    parent = list(range(len(ids)))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    hexes = list(h)
    n_bands = 16
    width = len(hexes[0]) // n_bands if hexes else 0
    for band in range(n_bands):
        buckets = defaultdict(list)
        for i, x in enumerate(hexes):
            buckets[x[band * width:(band + 1) * width]].append(i)
        for members in buckets.values():
            if len(members) < 2 or len(members) > 2000:
                continue
            for a_i in range(len(members)):
                for b_i in range(a_i + 1, len(members)):
                    a, b = members[a_i], members[b_i]
                    if bin(hashes[a] ^ hashes[b]).count("1") <= max_distance:
                        parent[find(a)] = find(b)
    roots = [find(i) for i in range(len(ids))]
    sizes = Counter(roots)
    out = pd.Series(np.nan, index=df.index)
    for i, r in enumerate(roots):
        if sizes[r] > 1:
            out.at[ids[i]] = r
    return out


def file_info(path) -> dict:
    """Read a file ONCE: SHA-1 of its bytes (exact duplicates), size, format, colour mode, dHash."""
    info = {"file_hash": None, "file_bytes": None, "width": None, "height": None,
            "file_format": None, "color_mode": None, "dhash": None, "readable": False, "error": None}
    try:
        data = Path(path).read_bytes()
        info["file_hash"] = hashlib.sha1(data).hexdigest()
        info["file_bytes"] = len(data)
        with Image.open(io.BytesIO(data)) as im:
            info.update(width=im.size[0], height=im.size[1], file_format=im.format, color_mode=im.mode)
            info["dhash"] = _dhash(im)
        info["readable"] = True
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info


def add_file_info(df: pd.DataFrame, path_col="image_path", workers=16) -> pd.DataFrame:
    """Run file_info on every row, in parallel threads (Drive is slow; threads hide the waiting)."""
    paths = df[path_col].tolist()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        infos = list(tqdm(ex.map(file_info, paths), total=len(paths), desc="Hashing + reading headers"))
    info_df = pd.DataFrame(infos, index=df.index)
    return pd.concat([df.drop(columns=[c for c in info_df.columns if c in df.columns]), info_df], axis=1)


def mark_duplicates(df: pd.DataFrame, label_col="label") -> pd.DataFrame:
    """Add `status`: ok | unreadable | duplicate | label_conflict.

    Files with identical bytes (same SHA-1) are the same image. We keep ONE copy
    (the first by image_id) and mark the rest 'duplicate', so the same image can
    never appear in two splits. If identical files carry different labels, every
    copy is 'label_conflict' and excluded.
    """
    df = df.copy()
    df["status"] = np.where(df["readable"], "ok", "unreadable")
    df["duplicate_of"] = None
    readable = df[df["readable"]]
    dup = readable[readable["file_hash"].duplicated(keep=False)].sort_values("image_id")
    for _, g in dup.groupby("file_hash"):
        if label_col in g and g[label_col].nunique() > 1:
            df.loc[g.index, "status"] = "label_conflict"
        else:
            df.loc[g.index[1:], "status"] = "duplicate"
            df.loc[g.index[1:], "duplicate_of"] = df.at[g.index[0], "image_id"]
    return df

**2.3 · `vs_genimage.py`**: GenImage manifest + ONE fixed leave-one-generator-out split (reused by M1, M2, Fusion).

In [ ]:
%%writefile vslib/vs_genimage.py
"""vs_genimage: GenImage manifest + ONE fixed leave-one-generator-out split (reused by M1, M2, Fusion).

Folder rules -> manifest -> candidate sampling -> split -> leakage checks -> save/load.
Labels: Real = 0, AI = 1.
"""
from __future__ import annotations

import hashlib
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

from vs_common import load_json, now, save_json
from vs_data import near_duplicate_groups

SHARED_REAL = "SHARED_REAL"          # generator name for a real-image folder not tied to one generator
SPLITS = ("train", "val", "test_seen", "test_unseen")
MANIFEST_COLUMNS = ["image_id", "image_path", "label", "generator", "file_hash",
                    "width", "height", "file_format", "split"]
EXTRA_COLUMNS = ["source_folder", "color_mode", "file_bytes", "dhash", "status", "duplicate_of", "sample_rank"]
AI_WORDS = {"ai", "fake", "fakes", "generated", "synthetic", "gen"}
REAL_WORDS = {"nature", "real", "natural", "authentic", "reals"}


def _label_guess(rel: str):
    """Guess 0/1 from folder words, checking the deepest folder first. Only a SUGGESTION."""
    for part in reversed(rel.split("/")):
        tokens = set(re.split(r"[^a-z0-9]+", part.lower()))
        if tokens & REAL_WORDS:
            return 0
        if tokens & AI_WORDS:
            return 1
    return None


def suggest_rules(folders: dict, depth: int = 3) -> None:
    """Print a FOLDER_RULES template. Generator names are NOT guessed; you type them."""
    groups = {}
    for rel, files in folders.items():
        parts = [] if rel == "." else rel.split("/")
        key = "/".join(parts[:depth]) if parts else "."
        groups[key] = groups.get(key, 0) + len(files)
    print("FOLDER_RULES = [")
    for key in sorted(groups):
        g = _label_guess(key)
        label = "CHANGE_ME" if g is None else g
        note = "label guessed from the folder name: CHECK it" if g is not None else "label unknown: set 0 or 1"
        lab = f'"{label}"' if label == "CHANGE_ME" else label
        print(f'    {{"folder": "{key}", "generator": "CHANGE_ME", "label": {lab}}},   # {groups[key]:,} images; {note}')
    print("]")


def _validate_rules(rules):
    if not rules:
        raise ValueError("FOLDER_RULES is empty. Copy the template printed in Section 4 and fill it in.")
    for r in rules:
        missing = {"folder", "generator", "label"} - set(r)
        if missing:
            raise ValueError(f"Rule {r} is missing {missing}.")
        if str(r["generator"]).strip() in ("", "CHANGE_ME") or r["label"] not in (0, 1):
            raise ValueError(f"Rule not filled in yet: {r}\nSet 'generator' to a name and 'label' to 0 (real) or 1 (AI).")
        if r["generator"] == SHARED_REAL and r["label"] != 0:
            raise ValueError(f"{SHARED_REAL} can only be used for real images (label 0): {r}")


def _matches(rel: str, rule_folder: str) -> bool:
    rf = rule_folder.strip("/")
    return rf in ("", ".") or rel == rf or rel.startswith(rf + "/")


def apply_rules(folders: dict, data_root, rules, ignore=()) -> pd.DataFrame:
    """One row per image: image_id (path relative to DATA_ROOT), image_path, label, generator.

    Every image folder must match a rule (the longest matching rule wins) or be listed
    in IGNORE_FOLDERS. Nothing is skipped silently.
    """
    _validate_rules(rules)
    data_root = Path(data_root)
    rows, unmatched, used = [], [], set()
    for rel, files in folders.items():
        if any(_matches(rel, ig) for ig in ignore):
            continue
        hits = [r for r in rules if _matches(rel, r["folder"])]
        if not hits:
            unmatched.append((rel, len(files)))
            continue
        r = max(hits, key=lambda r: len(r["folder"].strip("/")))
        used.add(r["folder"])
        for f in files:
            image_id = f if rel == "." else f"{rel}/{f}"
            rows.append({"image_id": image_id, "image_path": str(data_root / image_id),
                         "label": int(r["label"]), "generator": str(r["generator"]), "source_folder": rel})
    if unmatched:
        lines = "\n".join(f"  {rel}  ({n} images)" for rel, n in unmatched[:20])
        raise ValueError(f"{len(unmatched)} image folder(s) match no rule:\n{lines}\n"
                         "Add a rule for each one, or list it in IGNORE_FOLDERS on purpose.")
    for r in rules:
        if r["folder"] not in used:
            print(f"WARNING: rule matched no folder (typo?): {r}")
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No images matched the rules.")
    return df


def ai_generators(df) -> list:
    return sorted(df.loc[df["label"] == 1, "generator"].unique())


def sample_candidates(df: pd.DataFrame, n_per_class: int, seed: int, oversample: float = 1.3) -> pd.DataFrame:
    """Shuffle each (generator, label) group with the fixed seed and keep a few more than needed.

    Only these candidates are hashed (hashing the whole dataset would be slow); the
    extra ~30% replaces candidates later removed as duplicates or unreadable.
    """
    gens = ai_generators(df)
    parts = []
    for (gen, label), g in df.groupby(["generator", "label"]):
        want = n_per_class * (len(gens) if gen == SHARED_REAL else 1)
        k = min(len(g), math.ceil(want * oversample))
        cand = g.sort_values("image_id").sample(n=k, random_state=seed).copy()
        cand["sample_rank"] = np.arange(k)        # the random order used to pick images later
        parts.append(cand)
    return pd.concat(parts).reset_index(drop=True)


def _assign_fractions(df, idx, fractions):
    n = len(idx)
    n_train = int(round(fractions["train"] * n))
    n_val = int(round(fractions["val"] * n))
    df.loc[idx, "split"] = ["train"] * n_train + ["val"] * n_val + ["test_seen"] * (n - n_train - n_val)


def make_logo_split(df: pd.DataFrame, unseen: str, n_per_class: int, fractions: dict) -> pd.DataFrame:
    """Leave-one-generator-out split.

    Seen generators -> train / val / test_seen (70 / 15 / 15 by default), per generator and label.
    Unseen generator -> test_unseen ONLY (never train or val).
    SHARED_REAL real images (if used): n_per_class go to test_unseen, the rest are split like the seen pool.
    """
    gens = ai_generators(df)
    if unseen not in gens:
        raise ValueError(f"UNSEEN_GENERATOR = {unseen!r} is not one of the AI generators found: {gens}")
    if len(gens) < 2:
        raise ValueError(f"Leave-one-generator-out needs at least 2 AI generators; found {gens}")
    df = df.copy()
    df["split"] = np.where(df["status"] == "ok", "unused", "excluded_" + df["status"].astype(str))
    for (gen, label), g in df[df["status"] == "ok"].groupby(["generator", "label"]):
        g = g.sort_values("sample_rank")
        if gen == SHARED_REAL:
            need = n_per_class * len(gens)
            if len(g) < need:
                raise ValueError(f"{SHARED_REAL}: {len(g)} usable images, need {need}. Lower N_PER_CLASS.")
            chosen = g.iloc[:need]
            df.loc[chosen.index[:n_per_class], "split"] = "test_unseen"
            _assign_fractions(df, chosen.index[n_per_class:], fractions)
            continue
        if gen not in gens:
            raise ValueError(f"Real images are mapped to generator {gen!r}, which has no AI images. "
                             f"Use one of {gens} or {SHARED_REAL}.")
        if len(g) < n_per_class:
            raise ValueError(f"Only {len(g)} usable (readable, non-duplicate) images for generator={gen!r}, "
                             f"label={label}; N_PER_CLASS={n_per_class}. Lower N_PER_CLASS or add images.")
        chosen = g.iloc[:n_per_class]
        if gen == unseen:
            df.loc[chosen.index, "split"] = "test_unseen"
        else:
            _assign_fractions(df, chosen.index, fractions)
    return df


def drop_cross_split_near_duplicates(df: pd.DataFrame, enabled: bool = True):
    """Near-identical pictures (256-bit dHash within 12 bits) in DIFFERENT splits: keep the copy in the
    earliest split (train < val < test_seen < test_unseen) and exclude the others. Returns (df, report)."""
    order = {s: i for i, s in enumerate(SPLITS)}
    used = df[df["split"].isin(SPLITS)]
    groups = near_duplicate_groups(used)
    rows, drop = [], []
    for gid, g in used.assign(group=groups).dropna(subset=["group"]).groupby("group"):
        if g["split"].nunique() < 2:
            continue
        keep_split = min(g["split"], key=order.get)
        drop.extend(g[g["split"] != keep_split].index)
        rows.append({"splits": sorted(g["split"].unique(), key=order.get), "n": len(g),
                     "labels": sorted(g["label"].unique()), "image_ids": list(g["image_id"])[:4], "kept_in": keep_split})
    df = df.copy()
    if enabled and drop:
        df.loc[drop, "split"] = "excluded_near_duplicate"
    return df, pd.DataFrame(rows)


def split_counts(df: pd.DataFrame) -> pd.DataFrame:
    used = df[df["split"].isin(SPLITS)].copy()
    used["class"] = used["label"].map({0: "real", 1: "ai"})
    t = used.pivot_table(index="split", columns=["generator", "class"], values="image_id",
                         aggfunc="count", fill_value=0)
    t = t.reindex([s for s in SPLITS if s in t.index])
    t["TOTAL"] = t.sum(axis=1)
    return t


def check_leakage(df: pd.DataFrame, unseen: str, near_dup_enabled: bool = True) -> pd.DataFrame:
    """Run every leakage check; raise AssertionError if any FAILS. Nothing is skipped."""
    used = df[df["split"].isin(SPLITS)]
    checks = []

    def add(name, ok, detail="", level="FAIL"):
        checks.append({"check": name, "result": "PASS" if ok else level, "detail": detail})

    bad = used[used["split"].isin(["train", "val"]) & (used["generator"] == unseen)]
    add(f"Unseen generator '{unseen}' absent from train and val", bad.empty, f"{len(bad)} offending rows")
    bad = used[(used["split"] == "test_unseen") & (used["label"] == 1) & (used["generator"] != unseen)]
    add("test_unseen AI images come only from the unseen generator", bad.empty, f"{len(bad)} offending rows")
    bad = used[used["split"] == "test_seen"]
    add("test_seen contains no unseen-generator images", (bad["generator"] != unseen).all())
    per_hash = used.groupby("file_hash")["split"].nunique()
    add("No identical file (same SHA-1) in two splits", int((per_hash > 1).sum()) == 0,
        f"{int((per_hash > 1).sum())} shared hashes")
    add("No file hash used twice inside the experiment", not used["file_hash"].duplicated().any(),
        f"{int(used['file_hash'].duplicated().sum())} repeats")
    add("No image path used twice", not used["image_path"].duplicated().any())
    for s in SPLITS:
        labs = set(used.loc[used["split"] == s, "label"])
        add(f"'{s}' contains both Real and AI", labs == {0, 1}, f"labels present: {sorted(labs)}")
    groups = near_duplicate_groups(used)
    per_group = used.assign(group=groups).dropna(subset=["group"]).groupby("group")["split"].nunique()
    n_near = int((per_group > 1).sum())
    add("No near-identical image (dHash within 12 bits) in two splits", n_near == 0,
        f"{n_near} groups" + ("" if near_dup_enabled else " (DROP_NEAR_DUPLICATES is off)"),
        level="FAIL" if near_dup_enabled else "WARNING")
    report = pd.DataFrame(checks)
    if (report["result"] == "FAIL").any():
        print(report.to_string(index=False))
        raise AssertionError("LEAKAGE CHECK FAILED. Do not train on this split; see the table above.")
    return report


def split_name(unseen: str, n_per_class: int, seed: int) -> str:
    safe = re.sub(r"[^A-Za-z0-9._-]+", "-", unseen)
    return f"genimage_unseen-{safe}_n{n_per_class}_seed{seed}"


def split_frames(manifest: pd.DataFrame) -> dict:
    """{split: DataFrame} in a FIXED order (sorted by image_id). Row i of a split CSV = cache index i."""
    return {s: manifest[manifest["split"] == s].sort_values("image_id").reset_index(drop=True) for s in SPLITS}


def _fingerprint(ids) -> str:
    return hashlib.sha256("\n".join(ids).encode()).hexdigest()[:16]


def save_split(manifest: pd.DataFrame, split_dir, info: dict) -> dict:
    split_dir = Path(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)
    cols = MANIFEST_COLUMNS + [c for c in EXTRA_COLUMNS if c in manifest.columns]
    manifest[cols].sort_values("image_id").to_csv(split_dir / "manifest.csv", index=False)
    frames = split_frames(manifest)
    for s, f in frames.items():
        f[cols].to_csv(split_dir / f"{s}.csv", index=False)
    info = {**info, "created": now(),
            "counts": {s: {"total": len(f), "real": int((f["label"] == 0).sum()), "ai": int((f["label"] == 1).sum())}
                       for s, f in frames.items()},
            "fingerprints": {s: _fingerprint(f["image_id"].tolist()) for s, f in frames.items()}}
    save_json(info, split_dir / "split_info.json")
    return info


def load_split(split_dir, data_root=None):
    """Load a saved split and verify it was not changed. If DATA_ROOT differs (e.g. a teammate's Drive),
    image paths are rebuilt from image_id."""
    split_dir = Path(split_dir)
    info = load_json(split_dir / "split_info.json")
    manifest = pd.read_csv(split_dir / "manifest.csv", dtype={"image_id": str, "generator": str, "file_hash": str,
                                                             "dhash": str, "source_folder": str})
    if data_root is not None:
        manifest["image_path"] = [str(Path(data_root) / i) for i in manifest["image_id"]]
    frames = split_frames(manifest)
    for s, f in frames.items():
        if _fingerprint(f["image_id"].tolist()) != info["fingerprints"][s]:
            raise ValueError(f"Split '{s}' in {split_dir} does not match its saved fingerprint. Was a CSV edited?")
    return manifest, frames, info

**2.4 · `vs_preprocess.py`**: the ONE shared preprocessing pipeline + cached datasets (reused by M1, M2, Fusion).

In [ ]:
%%writefile vslib/vs_preprocess.py
"""vs_preprocess: the ONE shared preprocessing pipeline + cached datasets (reused by M1, M2, Fusion).

Every image, Real or AI:  RGB -> resize short side -> centre crop -> JPEG re-encode (bias control)
-> cached as uint8. Datasets return float tensors in 0..1 that are NOT normalised; each model
normalises inside itself (ImageNet stats for M1, CLIP stats for M2), so one cache serves all models.
"""
from __future__ import annotations

import hashlib
import io
import json
import os
import random
import shutil
import zlib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def jpeg_quality_for(image_id: str, quality_range, seed: int) -> int:
    """A JPEG quality drawn from the range, fixed per image (same image -> same quality on every run)."""
    rng = random.Random(zlib.crc32(f"{seed}:{image_id}".encode()))
    return rng.randint(int(quality_range[0]), int(quality_range[1]))


def jpeg_reencode(img: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    with Image.open(buf) as out:
        return out.convert("RGB")


def preprocess_base(img: Image.Image, image_id: str, pcfg: dict, seed: int):
    """Deterministic part of preprocessing. Returns (uint8 array [S, S, 3], info dict)."""
    img = img.convert("RGB")                                   # 1. RGB (drops alpha, expands greyscale)
    w, h = img.size
    target, S = pcfg["resize_short_side"], pcfg["cache_size"]
    scale = target / min(w, h)                                 # 2. shorter side -> `target` pixels
    new_w, new_h = max(S, round(w * scale)), max(S, round(h * scale))
    img = img.resize((new_w, new_h), Image.BICUBIC)
    left, top = (new_w - S) // 2, (new_h - S) // 2             # 3. centre crop S x S
    img = img.crop((left, top, left + S, top + S))
    q = jpeg_quality_for(image_id, pcfg["jpeg_quality_range"], seed)
    img = jpeg_reencode(img, q)                                # 4. JPEG bias control (same for both classes)
    return np.asarray(img, dtype=np.uint8), {"upscaled": min(w, h) < target, "jpeg_quality": q}


def _cache_fingerprint(image_ids, pcfg, seed) -> str:
    payload = json.dumps({"ids": list(image_ids), "pcfg": pcfg, "seed": seed}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def _cache_valid(npy: Path, fp: str) -> bool:
    meta = npy.with_suffix(".json")
    return npy.exists() and meta.exists() and json.loads(meta.read_text()).get("fingerprint") == fp


def build_cache(df, cache_file, pcfg: dict, seed: int, drive_copy=None, workers: int = 16) -> dict:
    """Preprocess every image in `df` (in row order) into one .npy file on the fast local disk.

    Reuses an existing valid cache; otherwise copies one from Drive if available; otherwise builds it
    (and saves a copy to Drive so the next Colab session can skip this step).
    """
    cache_file = Path(cache_file)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    fp = _cache_fingerprint(df["image_id"].tolist(), pcfg, seed)
    if _cache_valid(cache_file, fp):
        print(f"  cache OK (local): {cache_file.name}")
        return json.loads(cache_file.with_suffix(".json").read_text())
    if drive_copy is not None and _cache_valid(Path(drive_copy), fp):
        shutil.copy(drive_copy, cache_file)
        shutil.copy(Path(drive_copy).with_suffix(".json"), cache_file.with_suffix(".json"))
        print(f"  cache copied from Drive: {cache_file.name}")
        return json.loads(cache_file.with_suffix(".json").read_text())

    S = pcfg["cache_size"]
    tmp = cache_file.with_name(cache_file.stem + ".tmp.npy")
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(len(df), S, S, 3))
    infos = [None] * len(df)
    ids, paths = df["image_id"].tolist(), df["image_path"].tolist()

    def work(i):
        with Image.open(paths[i]) as im:
            a, info = preprocess_base(im, ids[i], pcfg, seed)
        arr[i] = a
        infos[i] = info

    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(tqdm(ex.map(work, range(len(df))), total=len(df), desc=f"Preprocessing {cache_file.stem}"))
    arr.flush()
    del arr
    os.replace(tmp, cache_file)
    meta = {"fingerprint": fp, "n": len(df), "size": S, "pcfg": pcfg, "seed": seed,
            "upscaled": [i["upscaled"] for i in infos], "jpeg_quality": [i["jpeg_quality"] for i in infos]}
    cache_file.with_suffix(".json").write_text(json.dumps(meta))
    if drive_copy is not None:
        Path(drive_copy).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(cache_file, drive_copy)
        shutil.copy(cache_file.with_suffix(".json"), Path(drive_copy).with_suffix(".json"))
    return meta


def random_rescale(a: np.ndarray, scale_range, min_size: int, s: float | None = None) -> np.ndarray:
    """Resize a square uint8 image by a random factor s (log-uniform in scale_range), bicubic.
    If the result is smaller than min_size, it is resized up to min_size (a second resample).
    Training-only augmentation: it gives EVERY image a random resampling trace, unrelated to its label."""
    if s is None:
        lo, hi = scale_range
        s = float(np.exp(random.uniform(np.log(lo), np.log(hi))))   # log-uniform: x0.5 as likely as x2
    n = max(1, round(a.shape[0] * s))
    img = Image.fromarray(np.asarray(a)).resize((n, n), Image.BICUBIC)
    if n < min_size:
        img = img.resize((min_size, min_size), Image.BICUBIC)
    return np.array(img)                                            # writable copy (PIL arrays are read-only)


class CachedImageDataset(Dataset):
    """Item i = row i of the split CSV: {"x": float [3, crop, crop] in 0..1, "label": 0/1, "idx": i}.

    train=True : [optional random rescale] + random crop + random horizontal flip (augmentation, training only)
    train=False: centre crop, no randomness (validation / test)
    rescale_range: e.g. (0.5, 2.0) turns on random rescaling; None = off (the baseline m1_run_001)
    """

    def __init__(self, df, cache_file, crop_size: int = 224, train: bool = False, hflip: bool = True,
                 rescale_range=None):
        self.labels = df["label"].astype(int).to_numpy()
        self.cache_file, self.crop, self.train, self.hflip = str(cache_file), crop_size, train, hflip
        self.rescale_range = tuple(rescale_range) if rescale_range else None
        self._arr = None
        n = np.load(self.cache_file, mmap_mode="r").shape[0]
        if n != len(df):
            raise ValueError(f"Cache {cache_file} has {n} images but the split has {len(df)}. Rebuild the cache.")

    def _array(self):
        if self._arr is None:                      # opened lazily: each DataLoader worker opens its own view
            self._arr = np.load(self.cache_file, mmap_mode="r")
        return self._arr

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        a = self._array()[i]                        # [S, S, 3] uint8
        if self.train and self.rescale_range:       # training only; val/test never reach this line
            a = random_rescale(a, self.rescale_range, self.crop)
        S, c = a.shape[0], self.crop
        if self.train:
            top, left = random.randint(0, S - c), random.randint(0, S - c)
        else:
            top = left = (S - c) // 2
        a = a[top:top + c, left:left + c]
        if self.train and self.hflip and random.random() < 0.5:
            a = a[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div_(255.0)
        return {"x": x, "label": torch.tensor(self.labels[i], dtype=torch.long), "idx": i}


def seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


def make_loader(ds, batch_size, shuffle, num_workers, seed, drop_last=False):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
                      pin_memory=torch.cuda.is_available(), worker_init_fn=seed_worker, generator=g,
                      drop_last=drop_last, persistent_workers=num_workers > 0)


def cached_image(cache_file, i, crop=None) -> np.ndarray:
    """One cached image as uint8 HWC (optionally centre-cropped), for plots."""
    a = np.load(str(cache_file), mmap_mode="r")[i]
    if crop:
        o = (a.shape[0] - crop) // 2
        a = a[o:o + crop, o:o + crop]
    return np.array(a)

**2.5 · `vs_features.py`**: input representations for the ResNet detectors.

In [ ]:
%%writefile vslib/vs_features.py
"""vs_features: input representations for the ResNet detectors.

  rgb : the image, normalised with ImageNet mean/std                  (3 channels)
  hp  : high-pass residual = image - Gaussian-blurred image           (3 channels)
  fft : centred log(1 + |FFT|) of the grey image                       (1 channel)

input_mode chooses the combination: "rgb" (baseline), "hp_fft" (proposal M1), "rgb_hp_fft" (optional).
Features are computed on the GPU inside the model, in float32 (FFT is not reliable in half precision).
"""
from __future__ import annotations

import contextlib
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)  # for M2 later
INPUT_MODES = {"rgb": ["rgb"], "hp_fft": ["hp", "fft"], "rgb_hp_fft": ["rgb", "hp", "fft"]}
PART_CHANNELS = {"rgb": 3, "hp": 3, "fft": 1}


def gaussian_kernel(size: int = 5, sigma: float = 1.0) -> torch.Tensor:
    ax = torch.arange(size, dtype=torch.float32) - (size - 1) / 2
    g = torch.exp(-(ax ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return torch.outer(g, g)                                   # [size, size], sums to 1


def high_pass(x01: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """Residual = image - blurred image. x01: [B, 3, H, W] in 0..1; kernel: [3, 1, k, k].

    Blurring keeps only slow, smooth changes (shapes, colours). Subtracting the blur leaves the
    fast changes: edges, fine texture, sensor noise and generator artefacts.
    """
    pad = kernel.shape[-1] // 2
    blurred = F.conv2d(F.pad(x01, (pad, pad, pad, pad), mode="reflect"), kernel, groups=3)
    return x01 - blurred


def fft_log_magnitude(x01: torch.Tensor) -> torch.Tensor:
    """[B, 3, H, W] -> [B, 1, H, W]: centred log-magnitude spectrum of the grey image.

    fft2 rewrites the image as a sum of waves; |.| is how strong each wave is; fftshift moves the
    slow (low-frequency) waves to the centre; log1p squeezes the huge range of values.
    Periodic generator artefacts appear as bright off-centre dots.
    """
    gray = (0.299 * x01[:, 0] + 0.587 * x01[:, 1] + 0.114 * x01[:, 2]).unsqueeze(1).float()
    spec = torch.fft.fftshift(torch.fft.fft2(gray, norm="ortho"), dim=(-2, -1))
    return torch.log1p(spec.abs())


def _no_autocast(device_type: str):
    if device_type in ("cuda", "cpu"):
        return torch.autocast(device_type=device_type, enabled=False)
    return contextlib.nullcontext()


class FeatureBuilder(nn.Module):
    """Turns a 0..1 image batch into the model input for the chosen input_mode."""

    def __init__(self, input_mode: str = "hp_fft", hp_kernel: int = 5, hp_sigma: float = 1.0):
        super().__init__()
        if input_mode not in INPUT_MODES:
            raise ValueError(f"input_mode must be one of {list(INPUT_MODES)}")
        self.input_mode, self.parts = input_mode, INPUT_MODES[input_mode]
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        k = gaussian_kernel(hp_kernel, hp_sigma)
        self.register_buffer("hp_kernel", k.expand(3, 1, hp_kernel, hp_kernel).clone())
        # Scale statistics, measured on TRAINING images only (fit_stats) and saved inside checkpoints.
        self.register_buffer("hp_std", torch.tensor(1.0))
        self.register_buffer("fft_mean", torch.tensor(0.0))
        self.register_buffer("fft_std", torch.tensor(1.0))
        self.register_buffer("stats_fitted", torch.tensor(input_mode == "rgb"))

    @property
    def n_channels(self) -> int:
        return sum(PART_CHANNELS[p] for p in self.parts)

    def raw_parts(self, x01) -> dict:
        """Un-normalised representations, for visualisation."""
        x = x01.float()
        return {"rgb": x, "hp": high_pass(x, self.hp_kernel), "fft": fft_log_magnitude(x)}

    def forward(self, x01: torch.Tensor) -> torch.Tensor:
        with _no_autocast(x01.device.type):
            x = x01.float()
            out = []
            for p in self.parts:
                if p == "rgb":
                    out.append((x - self.mean) / self.std)
                elif p == "hp":
                    out.append(high_pass(x, self.hp_kernel) / self.hp_std)
                else:
                    out.append((fft_log_magnitude(x) - self.fft_mean) / self.fft_std)
            return torch.cat(out, dim=1)

    @torch.no_grad()
    def fit_stats(self, loader, device, max_batches: int = 20) -> dict:
        """Measure the typical size of high-pass and FFT values on TRAINING batches, so the network
        sees inputs of roughly unit scale (residuals are tiny, about +/-0.05, otherwise)."""
        if self.input_mode == "rgb":
            return {}
        hp_sq = hp_n = f_sum = f_sq = f_n = 0.0
        for i, b in enumerate(loader):
            if i >= max_batches:
                break
            x = b["x"].to(device).float()
            if "hp" in self.parts:
                r = high_pass(x, self.hp_kernel)
                hp_sq += float((r ** 2).sum()); hp_n += r.numel()
            if "fft" in self.parts:
                f = fft_log_magnitude(x)
                f_sum += float(f.sum()); f_sq += float((f ** 2).sum()); f_n += f.numel()
        stats = {}
        if hp_n:
            stats["hp_std"] = max(math.sqrt(hp_sq / hp_n), 1e-6)
            self.hp_std.fill_(stats["hp_std"])
        if f_n:
            m = f_sum / f_n
            stats["fft_mean"], stats["fft_std"] = m, max(math.sqrt(max(f_sq / f_n - m * m, 0.0)), 1e-6)
            self.fft_mean.fill_(stats["fft_mean"]); self.fft_std.fill_(stats["fft_std"])
        self.stats_fitted.fill_(True)
        return stats

**2.6 · `vs_models.py`**: ResNet-50 detectors. The RGB baseline and M1 are the SAME class with a different input_mode.

In [ ]:
%%writefile vslib/vs_models.py
"""vs_models: ResNet-50 detectors. The RGB baseline and M1 are the SAME class with a different input_mode.

forward(x01) returns a dict:
  logits   [B]        raw score (input to BCEWithLogitsLoss)
  prob     [B]        P(AI) = sigmoid(logits)
  pred     [B]        predicted label (1 = AI if prob >= threshold, else 0 = Real)
  features [B, 2048]  pooled ResNet features, kept for the future Fusion model
"""
from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import ResNet50_Weights, resnet50

from vs_features import INPUT_MODES, FeatureBuilder


def adapt_conv1(old: nn.Conv2d, parts) -> nn.Conv2d:
    """New first layer for 4 or 7 input channels that still starts from ImageNet knowledge.

    rgb and hp channels copy the pretrained RGB filters; the fft channel uses their average.
    Weights are scaled by 3 / channels so the layer's output starts at a similar size.
    """
    w = old.weight.detach()                                    # [64, 3, 7, 7]
    blocks = [w.clone() if p in ("rgb", "hp") else w.mean(dim=1, keepdim=True) for p in parts]
    new_w = torch.cat(blocks, dim=1)
    new_w = new_w * (3.0 / new_w.shape[1])
    conv = nn.Conv2d(new_w.shape[1], old.out_channels, kernel_size=old.kernel_size,
                     stride=old.stride, padding=old.padding, bias=False)
    with torch.no_grad():
        conv.weight.copy_(new_w)
    return conv


class ResNetDetector(nn.Module):
    def __init__(self, input_mode="hp_fft", pretrained=True, dropout=0.3, hp_kernel=5, hp_sigma=1.0):
        super().__init__()
        self.input_mode = input_mode
        self.features_in = FeatureBuilder(input_mode, hp_kernel, hp_sigma)
        net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        self.conv1_adapted = input_mode != "rgb"
        if self.conv1_adapted:
            net.conv1 = adapt_conv1(net.conv1, INPUT_MODES[input_mode])
        net.fc = nn.Identity()                                 # remove the 1000-class ImageNet head
        self.backbone = net
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2048, 1))   # binary Real/AI head
        self.feature_dim = 2048

    def forward(self, x01: torch.Tensor, threshold: float = 0.5) -> dict:
        z = self.features_in(x01)                             # [B, C, H, W] model input
        feats = self.backbone(z)                              # [B, 2048]
        logits = self.head(feats).squeeze(1).float()          # [B]
        prob = torch.sigmoid(logits)
        return {"logits": logits, "prob": prob, "pred": (prob >= threshold).long(), "features": feats.float()}

    def param_groups(self, lr_backbone, lr_new, weight_decay):
        """Pretrained layers learn slowly (lr_backbone); new layers (head, adapted conv1) faster (lr_new)."""
        new = list(self.head.parameters())
        if self.conv1_adapted:
            new += list(self.backbone.conv1.parameters())
        new_ids = {id(p) for p in new}
        base = [p for p in self.backbone.parameters() if id(p) not in new_ids]
        return [{"params": base, "lr": lr_backbone, "weight_decay": weight_decay},
                {"params": new, "lr": lr_new, "weight_decay": weight_decay}]


def build_detector(input_mode: str, cfg: dict) -> ResNetDetector:
    m, f = cfg["model"], cfg["features"]
    return ResNetDetector(input_mode, pretrained=m["pretrained"], dropout=m["dropout"],
                          hp_kernel=f["hp_kernel"], hp_sigma=f["hp_sigma"])


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

**2.7 · `vs_metrics.py`**: classification metrics shared by M1, M2 and Fusion.

In [ ]:
%%writefile vslib/vs_metrics.py
"""vs_metrics: classification metrics shared by M1, M2 and Fusion.
(M3's watermark metrics will live in their own module, vs_wm_metrics, later.)

Convention: label 1 = AI (positive class), 0 = Real. Scores are P(AI).
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)

KEYS = ["accuracy", "auroc", "precision", "recall", "f1"]


def classification_metrics(labels, probs, threshold: float = 0.5) -> dict:
    y = np.asarray(labels).astype(int)
    p = np.asarray(probs, dtype=float)
    pred = (p >= threshold).astype(int)
    cm = confusion_matrix(y, pred, labels=[0, 1])               # [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = cm.ravel()
    return {
        "n": int(len(y)), "n_real": int((y == 0).sum()), "n_ai": int((y == 1).sum()), "threshold": threshold,
        "accuracy": float(accuracy_score(y, pred)),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan"),
        "precision": float(precision_score(y, pred, zero_division=0)),   # of images called AI, share truly AI
        "recall": float(recall_score(y, pred, zero_division=0)),         # of AI images, share caught
        "f1": float(f1_score(y, pred, zero_division=0)),
        "real_accuracy": float(tn / (tn + fp)) if (tn + fp) else float("nan"),
        "confusion_matrix": cm.tolist(),
    }


def metrics_by_generator(pred_df: pd.DataFrame) -> pd.DataFrame:
    """For each AI generator in the split: that generator's AI images vs ALL real images in the split."""
    rows = []
    real = pred_df[pred_df["label"] == 0]
    for g in sorted(pred_df.loc[pred_df["label"] == 1, "generator"].unique()):
        sub = pd.concat([real, pred_df[(pred_df["label"] == 1) & (pred_df["generator"] == g)]])
        m = classification_metrics(sub["label"], sub["p_ai"])
        rows.append({"ai_generator": g, "n_ai": m["n_ai"], "n_real": m["n_real"], "accuracy": m["accuracy"],
                     "auroc": m["auroc"], "ai_detection_rate": m["recall"], "real_accuracy": m["real_accuracy"]})
    return pd.DataFrame(rows)


def generalization_gap(seen: dict, unseen: dict) -> dict:
    """Seen minus unseen. Positive = performance DROPS on the unseen generator."""
    return {f"{k}_gap": seen[k] - unseen[k] for k in KEYS}

**2.8 · `vs_train.py`**: training, validation and prediction loops + sanity checks (shared by every classifier).

In [ ]:
%%writefile vslib/vs_train.py
"""vs_train: training, validation and prediction loops + sanity checks (shared by every classifier)."""
from __future__ import annotations

import contextlib
import random
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, Subset

from vs_common import load_checkpoint, now, save_checkpoint
from vs_metrics import classification_metrics
from vs_preprocess import make_loader


def autocast(device, amp):
    if amp["enabled"]:
        return torch.autocast(device_type=device.type, dtype=amp["dtype"])
    return contextlib.nullcontext()


def make_scaler(amp):
    return torch.amp.GradScaler("cuda", enabled=amp["use_scaler"])


def cosine_with_warmup(optimizer, total_steps, warmup_steps):
    """Learning rate rises linearly during warm-up, then follows a cosine curve down to 0."""
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def train_one_epoch(model, loader, optimizer, scheduler, scaler, device, amp, max_batches=None):
    model.train()
    loss_fn = nn.BCEWithLogitsLoss()
    total, n, correct, t0 = 0.0, 0, 0, time.time()
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x = batch["x"].to(device, non_blocking=True)
        y = batch["label"].to(device).float()
        with autocast(device, amp):
            out = model(x)
        loss = loss_fn(out["logits"], y)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Loss became {loss.item()} at batch {i}. Lower the learning rate.")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()          # with the scaler disabled this is a normal backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        total += loss.item() * len(y)
        n += len(y)
        correct += int((out["pred"] == y.long()).sum())
    return {"loss": total / max(n, 1), "acc": correct / max(n, 1), "n": n, "seconds": time.time() - t0}


@torch.no_grad()
def predict(model, loader, device, amp, max_batches=None):
    """Returns (DataFrame idx/label/logit/p_ai/pred sorted by idx, mean BCE loss)."""
    model.eval()
    loss_fn = nn.BCEWithLogitsLoss(reduction="sum")
    parts, loss_sum, n = [], 0.0, 0
    for i, b in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x = b["x"].to(device, non_blocking=True)
        y = b["label"].to(device)
        with autocast(device, amp):
            out = model(x)
        loss_sum += float(loss_fn(out["logits"], y.float()))
        n += len(y)
        parts.append(pd.DataFrame({"idx": b["idx"].numpy(), "label": b["label"].numpy(),
                                   "logit": out["logits"].float().cpu().numpy(),
                                   "p_ai": out["prob"].float().cpu().numpy(),
                                   "pred": out["pred"].cpu().numpy()}))
    df = pd.concat(parts).sort_values("idx").reset_index(drop=True)
    return df, loss_sum / max(n, 1)


def predictions_with_meta(pred_df, split_df, split, model_name):
    """Attach image_id, image_path, generator to predictions (row idx of the split CSV)."""
    meta = split_df.iloc[pred_df["idx"].to_numpy()][["image_id", "image_path", "generator", "label"]].reset_index(drop=True)
    if not (meta["label"].to_numpy() == pred_df["label"].to_numpy()).all():
        raise RuntimeError("Prediction rows do not line up with the split CSV. Rebuild the cache.")
    out = pd.concat([meta, pred_df[["logit", "p_ai", "pred"]].reset_index(drop=True)], axis=1)
    out.insert(0, "split", split)
    out["model"] = model_name
    out["correct"] = out["pred"] == out["label"]
    out["cache_index"] = pred_df["idx"].to_numpy()
    return out


def fit(model, train_loader, val_loader, cfg, run_dir, device, amp, resume=True, log=print, stats_loader=None):
    """Train with validation after every epoch. best.pt = highest val AUROC; last.pt = resume point.
    stats_loader: where the feature scale statistics are measured (default: train_loader). With random
    rescaling, pass the un-augmented training loader so the statistics match what val/test images look like."""
    tc, run_dir = cfg["train"], Path(run_dir)
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    optimizer = torch.optim.AdamW(model.param_groups(tc["lr_backbone"], tc["lr_new"], tc["weight_decay"]))
    steps = len(train_loader) if tc.get("max_train_batches") is None else min(len(train_loader), tc["max_train_batches"])
    scheduler = cosine_with_warmup(optimizer, tc["epochs"] * steps, tc["warmup_epochs"] * steps)
    scaler = make_scaler(amp)
    history, start, best_auroc, bad = [], 0, -1.0, 0

    if resume and last_path.exists():
        ck = load_checkpoint(last_path, device)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        history, start = ck["history"], ck["epoch"] + 1
        best_auroc, bad = ck["best_val_auroc"], ck["epochs_without_improvement"]
        if ck.get("finished"):
            log(f"Training already finished (epoch {ck['epoch'] + 1}); loaded it. Delete last.pt to retrain.")
            return pd.DataFrame(history)
        log(f"Resuming from epoch {start + 1}")
    else:
        stats = model.features_in.fit_stats(stats_loader or train_loader, device, cfg["features"]["stats_batches"])
        if stats:
            log(f"Feature scale statistics (training images only): {stats}")

    for epoch in range(start, tc["epochs"]):
        tr = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, device, amp, tc.get("max_train_batches"))
        pv, vloss = predict(model, val_loader, device, amp, tc.get("max_val_batches"))
        vm = classification_metrics(pv["label"], pv["p_ai"])
        row = {"epoch": epoch + 1, "train_loss": tr["loss"], "train_acc": tr["acc"], "val_loss": vloss,
               "val_acc": vm["accuracy"], "val_auroc": vm["auroc"],
               "lr_backbone": optimizer.param_groups[0]["lr"], "lr_new": optimizer.param_groups[1]["lr"],
               "train_seconds": round(tr["seconds"], 1)}
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        score = vm["auroc"] if not math.isnan(vm["auroc"]) else -1.0
        improved = score > best_auroc + 1e-6
        if improved:
            best_auroc, bad = score, 0
            save_checkpoint(best_path, {"model": model.state_dict(), "epoch": epoch + 1, "val_metrics": vm,
                                        "input_mode": model.input_mode, "config": cfg, "created": now()})
        else:
            bad += 1
        finished = bad >= tc["early_stop_patience"] or epoch + 1 == tc["epochs"]
        save_checkpoint(last_path, {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                                    "epoch": epoch, "history": history, "best_val_auroc": best_auroc,
                                    "epochs_without_improvement": bad, "finished": finished,
                                    "input_mode": model.input_mode, "config": cfg, "created": now()})
        log(f"epoch {epoch + 1:>2}/{tc['epochs']}  train_loss {tr['loss']:.4f}  train_acc {tr['acc']:.3f}  "
            f"val_loss {vloss:.4f}  val_acc {vm['accuracy']:.3f}  val_auroc {vm['auroc']:.4f}"
            f"{'  * best' if improved else ''}  ({tr['seconds']:.0f}s)")
        if bad >= tc["early_stop_patience"]:
            log(f"Early stopping: val AUROC has not improved for {bad} epochs.")
            break
    return pd.DataFrame(history)


# ---------------- sanity checks ----------------
class ShuffledLabels(Dataset):
    """Same images with RANDOM labels that carry exactly zero information about the true class:
    inside each true class, half the images get label 1 and half label 0 (random choice).
    mirror=True flips every random label (0 <-> 1)."""

    def __init__(self, base, seed, mirror=False):
        self.base = base
        true = base.labels if not isinstance(base, Subset) else base.dataset.labels[base.indices]
        rng = np.random.default_rng(seed)
        self.labels = np.zeros(len(true), dtype=int)
        for c in (0, 1):
            idx = np.where(true == c)[0]
            rng.shuffle(idx)
            self.labels[idx[len(idx) // 2:]] = 1
        if mirror:
            self.labels = 1 - self.labels

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        item = dict(self.base[i])
        item["label"] = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return item


def overfit_test(make_model, dataset, device, amp, steps, batch_size, seed) -> dict:
    """Can the model memorise a tiny set? If not, something is broken (labels, inputs, optimiser)."""
    model = make_model().to(device)
    loader = make_loader(dataset, batch_size, True, 0, seed)
    model.features_in.fit_stats(loader, device, max_batches=4)
    opt = torch.optim.AdamW(model.param_groups(3e-4, 3e-3, 0.0))
    scaler, step = make_scaler(amp), 0
    while step < steps:
        for b in loader:
            model.train()
            x, y = b["x"].to(device), b["label"].to(device).float()
            with autocast(device, amp):
                out = model(x)
            loss = nn.functional.binary_cross_entropy_with_logits(out["logits"], y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            step += 1
            if step >= steps:
                break
    pv, ploss = predict(model, make_loader(dataset, batch_size, False, 0, seed), device, amp)
    acc = float((pv["pred"] == pv["label"]).mean())
    return {"images": len(dataset), "steps": steps, "final_loss": ploss, "accuracy": acc,
            "passed": acc >= 0.9}


def shuffled_label_test(make_model, train_ds, val_loader, cfg, device, amp, seed) -> dict:
    """Random-label test, done as a MIRRORED PAIR.

    Model A trains on random labels L, model B on the mirror labels 1 - L. Neither label set says anything
    about the true class. Both start from the same weights, with the final layer set to exactly zero, and
    see the same images in the same order. If the model learns ONLY from the labels we give it, B is then
    an exact mirror of A (B's logit = -A's logit), so AUROC_B = 1 - AUROC_A and their mean is 0.5.
    If the TRUE label reaches training by another route (a bug in the label path), the mirror breaks and
    the mean moves away from 0.5.
    (One random-label model alone is not enough: when classes are easy to separate, its AUROC lands near
    0 or 1 depending on its random starting weights, even with no leakage.)
    Note: duplicate images across splits are checked in Section 7, not by this test.
    """
    sc, tc = cfg["sanity"], cfg["train"]
    n = min(sc["shuffled_train_images"], len(train_ds))
    rng = np.random.default_rng(seed)
    subset = Subset(train_ds, sorted(rng.choice(len(train_ds), size=n, replace=False).tolist()))
    aurocs, m = [], None
    for mirror in (False, True):
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)   # identical randomness for A and B
        loader = make_loader(ShuffledLabels(subset, seed, mirror), tc["active_batch_size"], True, 0, seed)
        model = make_model().to(device)
        nn.init.zeros_(model.head[1].weight)                  # zero final layer: no starting preference
        nn.init.zeros_(model.head[1].bias)
        model.features_in.fit_stats(loader, device, max_batches=4)
        opt = torch.optim.AdamW(model.param_groups(tc["lr_backbone"], tc["lr_new"], tc["weight_decay"]))
        scaler = make_scaler(amp)
        for _ in range(sc["shuffled_epochs"]):
            train_one_epoch(model, loader, opt, None, scaler, device, amp)
        pv, _ = predict(model, val_loader, device, amp)
        m = classification_metrics(pv["label"], pv["p_ai"])
        aurocs.append(m["auroc"])
        del model, opt
    mean = float(np.mean(aurocs))
    margin = 0.10        # a clean pipeline gives ~0.5 exactly; GPU rounding can move it slightly
    return {"train_images": n, "epochs": sc["shuffled_epochs"], "auroc_A": aurocs[0], "auroc_B_mirror": aurocs[1],
            "mean_auroc": mean, "band": (0.5 - margin, 0.5 + margin), "passed": abs(mean - 0.5) <= margin}

**2.9 · `vs_plots.py`**: shared plots (saved as PNG and shown in the notebook).

In [ ]:
%%writefile vslib/vs_plots.py
"""vs_plots: shared plots (saved as PNG and shown in the notebook)."""
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_curve, roc_auc_score

from vs_preprocess import cached_image

REAL_C, AI_C = "#2b8a3e", "#1f5fbf"


def _finish(fig, path):
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_history(h, title, path=None):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(h["epoch"], h["train_loss"], "o-", label="train loss")
    ax[0].plot(h["epoch"], h["val_loss"], "o-", label="validation loss")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("BCE loss"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h["epoch"], h["val_auroc"], "o-", color="purple", label="validation AUROC")
    ax[1].plot(h["epoch"], h["val_acc"], "s--", color="gray", label="validation accuracy")
    best = h["val_auroc"].idxmax()
    ax[1].axvline(h.loc[best, "epoch"], color="purple", alpha=.3, ls=":", label=f"best epoch {int(h.loc[best, 'epoch'])}")
    ax[1].set_ylim(0, 1.02); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.suptitle(title)
    _finish(fig, path)


def plot_roc(curves: dict, title, path=None):
    fig, ax = plt.subplots(figsize=(5.2, 5))
    for name, (y, p) in curves.items():
        if len(np.unique(y)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, label=f"{name} (AUROC {roc_auc_score(y, p):.3f})")
    ax.plot([0, 1], [0, 1], "k:", alpha=.5, label="chance")
    ax.set_xlabel("False positive rate (real called AI)"); ax.set_ylabel("True positive rate (AI caught)")
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=.3)
    _finish(fig, path)


def plot_confusion(cm, title, path=None):
    cm = np.asarray(cm)
    fig, ax = plt.subplots(figsize=(3.9, 3.5))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            pct = cm[i, j] / max(cm[i].sum(), 1) * 100
            ax.text(j, i, f"{cm[i, j]}\n({pct:.0f}%)", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["pred Real", "pred AI"]); ax.set_yticks([0, 1], ["true Real", "true AI"])
    ax.set_title(title, fontsize=10)
    _finish(fig, path)


def plot_score_hist(pred_df, title, path=None):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    bins = np.linspace(0, 1, 26)
    ax.hist(pred_df.loc[pred_df.label == 0, "p_ai"], bins=bins, alpha=.6, color=REAL_C, label="true Real")
    ax.hist(pred_df.loc[pred_df.label == 1, "p_ai"], bins=bins, alpha=.6, color=AI_C, label="true AI")
    ax.axvline(0.5, color="k", ls=":", label="threshold 0.5")
    ax.set_xlabel("P(AI)"); ax.set_ylabel("images"); ax.set_title(title); ax.legend()
    _finish(fig, path)


def plot_examples(pred_df, cache_file, correct: bool, title, path=None, n=12, seed=0):
    sub = pred_df[pred_df["correct"] == correct]
    if sub.empty:
        print(f"No {'correct' if correct else 'incorrect'} predictions to show for: {title}")
        return
    if correct:
        sub = sub.sample(min(n, len(sub)), random_state=seed)
    else:                                                    # most confident mistakes first
        sub = sub.assign(conf=(sub["p_ai"] - 0.5).abs()).sort_values("conf", ascending=False).head(n)
    cols = 6
    rows = int(np.ceil(len(sub) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.9 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, sub.iterrows()):
        ax.imshow(cached_image(cache_file, int(r["cache_index"]), crop=224))
        ax.set_title(f"true {'AI' if r['label'] == 1 else 'Real'} | P(AI) {r['p_ai']:.2f}\n{str(r['generator'])[:18]}",
                     fontsize=7, color="green" if correct else "red")
    fig.suptitle(title)
    _finish(fig, path)


def plot_before_after(rows, cache_file, path=None):
    """rows: DataFrame rows with image_path, label, generator and the cache index column 'cache_index'."""
    fig, axes = plt.subplots(2, len(rows), figsize=(2.3 * len(rows), 5), squeeze=False)
    for j, (_, r) in enumerate(rows.iterrows()):
        with Image.open(r["image_path"]) as im:
            axes[0][j].imshow(im.convert("RGB"))
            axes[0][j].set_title(f"{'AI' if r['label'] == 1 else 'Real'} | {str(r['generator'])[:14]}\n"
                                 f"original {im.size[0]}x{im.size[1]} {im.format}", fontsize=7)
        axes[1][j].imshow(cached_image(cache_file, int(r["cache_index"]), crop=224))
        axes[1][j].set_title("after: 224x224, JPEG re-encoded", fontsize=7)
        axes[0][j].axis("off"); axes[1][j].axis("off")
    fig.suptitle("Before (top) and after (bottom) shared preprocessing")
    _finish(fig, path)


def _to_img(t):
    return t.detach().cpu().permute(1, 2, 0).numpy() if t.shape[0] == 3 else t[0].detach().cpu().numpy()


def plot_representations(x01, labels, generators, feature_builder, path=None):
    """Rows = images; columns = RGB, high-pass residual (amplified), FFT log-magnitude."""
    parts = feature_builder.raw_parts(x01)
    n = len(x01)
    fig, axes = plt.subplots(n, 3, figsize=(8.4, 2.7 * n), squeeze=False)
    for i in range(n):
        hp = parts["hp"][i].abs().mean(0)
        hp = (hp / (hp.flatten().quantile(0.99) + 1e-8)).clamp(0, 1)
        axes[i][0].imshow(_to_img(parts["rgb"][i]))
        axes[i][1].imshow(hp.cpu().numpy(), cmap="magma")
        f = _to_img(parts["fft"][i])
        axes[i][2].imshow(f, cmap="viridis", vmin=np.percentile(f, 1), vmax=np.percentile(f, 99.5))
        axes[i][0].set_ylabel(f"{'AI' if labels[i] == 1 else 'Real'}\n{str(generators[i])[:16]}", fontsize=8)
        for j, t in enumerate(["RGB", "High-pass |residual|", "FFT log-magnitude"]):
            axes[i][j].set_xticks([]); axes[i][j].set_yticks([])
            if i == 0:
                axes[i][j].set_title(t)
    _finish(fig, path)


def plot_mean_spectra(spectra: dict, path=None):
    """spectra: {(generator, label): mean log-spectrum [H, W]} -> one panel each."""
    keys = sorted(spectra)
    fig, axes = plt.subplots(1, len(keys), figsize=(2.9 * len(keys), 3.2), squeeze=False)
    vmin = min(float(np.percentile(v, 1)) for v in spectra.values())
    vmax = max(float(np.percentile(v, 99.5)) for v in spectra.values())
    for ax, k in zip(axes[0], keys):
        ax.imshow(spectra[k], cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_title(f"{str(k[0])[:16]}\n{'AI' if k[1] == 1 else 'Real'}", fontsize=8)
        ax.axis("off")
    fig.suptitle("Average FFT log-magnitude per generator and class (training images)")
    _finish(fig, path)

In [ ]:
# Import the library (reload makes edits take effect without restarting Colab)
sys.path.insert(0, os.path.abspath("vslib"))
import vs_common, vs_data, vs_genimage, vs_preprocess, vs_features, vs_models, vs_metrics, vs_train, vs_plots
for m in (vs_common, vs_data, vs_genimage, vs_preprocess, vs_features, vs_models, vs_metrics, vs_train, vs_plots):
    importlib.reload(m)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)

# Save a copy of the library next to the project on Drive (M2 / Fusion / M3 notebooks import it from there)
lib_drive = Path(CONFIG["paths"]["lib_drive"])
lib_drive.mkdir(parents=True, exist_ok=True)
for f in Path("vslib").glob("*.py"):
    shutil.copy(f, lib_drive / f.name)
print("Library imported and copied to", lib_drive)

**Expected output:** `Mounted at /content/drive`, then nine `Writing vslib/...` lines, then `Library imported and copied to .../VeriStamp/vslib`.

## 3 · GPU check
**Purpose:** choose the device (GPU if available, otherwise CPU) and mixed precision.
- **A100:** bfloat16. Fast, and needs no gradient scaling.
- **Older GPUs (T4):** float16 with a GradScaler.
- **CPU:** full precision, small batches. Fine for the checks, not for training.

In [ ]:
vs_common.set_seed(SEED)
DEVICE = vs_common.get_device()
AMP = vs_common.amp_settings(DEVICE)
ENV = vs_common.environment_info()
CONFIG["train"]["active_batch_size"] = CONFIG["train"]["batch_size"] if DEVICE.type == "cuda" else CONFIG["train"]["cpu_batch_size"]
if DEVICE.type != "cuda":
    CONFIG["train"]["num_workers"] = 0
for k, v in ENV.items():
    print(f"{k:>15}: {v}")
print(f"{'mixed precision':>15}: {AMP['enabled']} ({AMP['dtype']})")
print(f"{'batch size':>15}: {CONFIG['train']['active_batch_size']}")
if DEVICE.type != "cuda":
    print("\nNOTE: no CUDA GPU. Everything up to the sanity checks works on CPU, but real training needs the A100.")

**Expected output on an A100:** `cuda_available: True`, `gpu: NVIDIA A100-SXM4-40GB` (or 80GB), `mixed precision: True (torch.bfloat16)`, `batch size: 64`.

## 4 · Dataset inspection
**Purpose:** see what is really in `DATA_ROOT` *before* assigning any labels. Nothing is assumed about the folder structure, and nothing is changed.

First, this section checks that **Section 0 verified the dataset** (`DATASET_SETUP.json`) and that `N_PER_CLASS` is achievable. Generators you did not select in Section 0 are not scanned.

What each step shows:
- **Tree:** how folders are nested, with image counts. Each count includes sub-folders.
- **Folder stats:** file format, size and colour mode per folder. This is the bias check: if real images are JPEGs and AI images are PNGs, a model can cheat by learning the format.
- **Shared file names:** many identical names in two *real* folders suggests the same photo appears twice, which is a leakage risk.
- **Folder rules:** the folder → (generator, label) mapping you confirmed in Section 0, with image counts. Section 5 uses it as is.

In [ ]:
assert "CHANGE_ME" not in DATA_ROOT, "Set DATA_ROOT in Section 1 first (copy it from the end of Section 0)."
assert Path(DATA_ROOT).exists(), f"DATA_ROOT not found: {DATA_ROOT}. Copy the DATA_ROOT line printed by Section 0."

# --- Gate: Section 0 must have verified this dataset ---
_setup_file = Path(DATA_ROOT) / "DATASET_SETUP.json"
assert _setup_file.exists(), ("DATASET_SETUP.json not found in DATA_ROOT. Run Section 0 (Dataset Acquisition and Setup) "
                              "until its last cell says DATASET SETUP VERIFIED.")
DATASET_SETUP = json.loads(_setup_file.read_text())
assert DATASET_SETUP.get("verified") is True, "Section 0 did not verify this dataset. Rerun Section 0."
assert N_PER_CLASS <= DATASET_SETUP["max_n_per_class"], (
    f"N_PER_CLASS = {N_PER_CLASS}, but Section 0 found only {DATASET_SETUP['max_n_per_class']} images for the smallest "
    "generator/class. Lower N_PER_CLASS in Section 1, or extract more images and rerun Section 0.")
CONFIG["data"]["selected_generators"] = DATASET_SETUP["selected_generators"]
CONFIG["data"]["dataset_subset_level"] = DATASET_SETUP["subset_level"]
print("Section 0 verified this dataset on", DATASET_SETUP["created"])
print("Selected generators:", DATASET_SETUP["selected_generators"], "| subset level:", DATASET_SETUP["subset_level"])
print("Folders not scanned (not selected / left out on purpose):", DATASET_SETUP["ignore_folders"], "\n")

t0 = time.time()
FOLDERS, OTHER_FILES = vs_data.scan_image_folders(DATA_ROOT, skip=DATASET_SETUP["ignore_folders"])
n_total = sum(len(v) for v in FOLDERS.values())
print(f"Scanned in {time.time() - t0:.0f}s: {n_total:,} images in {len(FOLDERS)} folders")
print("Non-image files:", dict(OTHER_FILES.most_common(10)) or "none")

In [ ]:
vs_data.print_tree(FOLDERS, root_label=Path(DATA_ROOT).name, max_depth=4)

In [ ]:
FOLDER_STATS = vs_data.folder_stats(DATA_ROOT, FOLDERS, per_folder=20)
print(FOLDER_STATS.to_string(index=False))

In [ ]:
n_shared, top_pairs = vs_data.shared_name_report(FOLDERS)
print(f"File names that appear in more than one folder: {n_shared:,}")
for (a, b), n in top_pairs:
    print(f"{n:>8,}  {a}   <->   {b}")
print("\n(Generic names such as 0001.png repeat by chance. Section 6 checks real duplicates by file CONTENT.)")

In [ ]:
# The folder -> (generator, label) rules were confirmed in Section 0; Section 5 loads them automatically.
print("Folder rules confirmed in Section 0 (Section 5 uses these; nothing to paste):\n")
_lab = {0: "Real", 1: "AI"}
for r in DATASET_SETUP["folder_rules"]:
    n = sum(len(v) for k, v in FOLDERS.items() if k == r["folder"] or k.startswith(r["folder"] + "/"))
    print(f'  {r["folder"]:<60} -> generator {r["generator"]!r:<26} label {r["label"]} ({_lab[r["label"]]}), {n:,} images')
_unmatched = [k for k in FOLDERS if not any(k == r["folder"] or k.startswith(r["folder"] + "/")
                                            for r in DATASET_SETUP["folder_rules"])]
print("\nImage folders without a rule:", _unmatched or "none")
inspection = {"created": vs_common.now(), "data_root": DATA_ROOT, "total_images": n_total,
              "non_image_files": dict(OTHER_FILES), "image_counts": {k: len(v) for k, v in FOLDERS.items()},
              "folder_stats": FOLDER_STATS.to_dict("records")}
vs_common.save_json(inspection, Path(CONFIG["paths"]["manifests_dir"]) / "inspection_report.json")
print("\nSaved inspection_report.json")

**Expected output:**
- a total image count and folder count
- a tree such as `<generator folder>/ ... ai/ 4,000 images`, `nature/ 4,000 images` (your names will differ)
- a stats table per folder
- the shared-name report
- the Section 0 folder rules, every `.../ai` folder with label 1 (AI) and every `.../nature` folder with label 0 (Real), and "Image folders without a rule: none"

📤 **Send me:** the scan line, the tree (first ~60 lines), the stats table, and the folder rules.

## 5 · Manifest creation
**Purpose:** build the project-wide manifest, one row per image: `image_id, image_path, label (Real = 0, AI = 1), generator, file_hash, width, height, file_format, split`, plus a few extra columns.

✏️ **Usually nothing to edit here:** if `FOLDER_RULES` is left empty, the mapping you confirmed in **Section 0** is loaded automatically, along with the folders it leaves out. Fill it in only to override Section 0. The format is:
- `folder`: a folder path relative to `DATA_ROOT`. It covers that folder and everything inside it. If two rules match, the longer (more specific) one wins.
- `generator`: **your** name for the generator, for example the folder name. Real images that belong to a generator's folder get **that generator's name**, so they follow it into the right split. If there is one real-image folder shared by all generators, use `"SHARED_REAL"`.
- `label`: `0` = Real, `1` = AI.
- Any image folder without a rule causes an **error**, so nothing is skipped silently. Put folders you deliberately don't want in `IGNORE_FOLDERS`.

To keep this fast, only a seeded random **sample** of each (generator, label) group is hashed: `N_PER_CLASS` images plus 30% spares.

In [ ]:
# ╔═══════════════════════════  <<< CHANGE  ═════════════════════════════╗
# ║ Leave EMPTY to use the rules confirmed in Section 0 (recommended).    ║
# ╚══════════════════════════════════════════════════════════════════════╝
FOLDER_RULES = [
    # {"folder": "path/relative/to/DATA_ROOT", "generator": "name_you_choose", "label": 1},   # AI images
    # {"folder": "path/relative/to/DATA_ROOT", "generator": "name_you_choose", "label": 0},   # real images
]
IGNORE_FOLDERS = []   # folders to leave out ON PURPOSE (each one is printed below)

# If you leave FOLDER_RULES empty, the mapping you confirmed in Section 0 is used:
if not FOLDER_RULES:
    FOLDER_RULES = DATASET_SETUP["folder_rules"]
    IGNORE_FOLDERS = sorted(set(IGNORE_FOLDERS) | set(DATASET_SETUP["ignore_folders"]))
    print(f"Using the {len(FOLDER_RULES)} folder rules confirmed in Section 0 (DATASET_SETUP.json).")

CONFIG["data"]["folder_rules"] = FOLDER_RULES
CONFIG["data"]["ignore_folders"] = IGNORE_FOLDERS
ALL_IMAGES = vs_genimage.apply_rules(FOLDERS, DATA_ROOT, FOLDER_RULES, IGNORE_FOLDERS)
for ig in IGNORE_FOLDERS:
    print("Ignored on purpose:", ig)
print(f"{len(ALL_IMAGES):,} images labelled by the rules\n")
print(ALL_IMAGES.pivot_table(index="generator", columns="label", values="image_id", aggfunc="count", fill_value=0)
      .rename(columns={0: "real (0)", 1: "ai (1)"}))

In [ ]:
CANDIDATES = vs_genimage.sample_candidates(ALL_IMAGES, N_PER_CLASS, SEED, CONFIG["data"]["oversample"])
print(f"Hashing {len(CANDIDATES):,} sampled candidates (reads each file once from Drive)...")
CANDIDATES = vs_data.add_file_info(CANDIDATES, workers=CONFIG["data"]["hash_workers"])
bad = CANDIDATES[~CANDIDATES["readable"]]
print(f"Unreadable files: {len(bad)}")
if len(bad):
    print(bad[["image_id", "error"]].head(10).to_string(index=False))

In [ ]:
# Bias evidence BEFORE preprocessing: do Real and AI files differ in format or size?
CANDIDATES["class"] = CANDIDATES["label"].map({0: "real", 1: "ai"})
print("File formats by generator and class:")
print(CANDIDATES.pivot_table(index=["generator", "class"], columns="file_format", values="image_id",
                             aggfunc="count", fill_value=0))
print("\nMedian width x height by generator and class:")
print(CANDIDATES.groupby(["generator", "class"])[["width", "height"]].median())
print("\nManifest preview:")
print(CANDIDATES[["image_id", "label", "generator", "file_hash", "width", "height", "file_format"]].head(5).to_string(index=False))

**Expected output:**
- a table of image counts per generator and class
- a hashing progress bar
- `Unreadable files: 0` (ideally)
- a format table and a size table
- a short manifest preview

**What to look for:** if one class is all PNG and the other all JPEG, or the sizes differ a lot, that is exactly the shortcut the JPEG step in Section 8 removes. Note it for the report.

📤 **Send me:** the counts table, the unreadable count, and both bias tables.

## 6 · Duplicate checking
**Purpose:** make sure the *same picture* can never be in two splits.
- **Exact duplicates:** identical bytes, so an identical SHA-1 `file_hash`. One copy is kept (the first by `image_id`); the others become `duplicate`. If identical files have *different* labels, every copy becomes `label_conflict` and is excluded.
- **Near duplicates:** the same picture re-saved, recompressed or resized has different bytes but an almost identical 256-bit perceptual hash (`dhash`, at most 12 bits different). These are reported here, and removed across splits in Section 7.

GenImage's real images come from ImageNet, so the same real photo may sit in two generator folders. This is the main leakage risk for the unseen-generator test.

In [ ]:
CANDIDATES = vs_data.mark_duplicates(CANDIDATES)
print("Status of sampled candidates:")
print(CANDIDATES["status"].value_counts().to_string())
dups = CANDIDATES[CANDIDATES["status"] == "duplicate"]
if len(dups):
    print("\nExamples of exact duplicates (removed copy -> kept copy):")
    print(dups[["image_id", "duplicate_of"]].head(10).to_string(index=False))
conf = CANDIDATES[CANDIDATES["status"] == "label_conflict"]
if len(conf):
    print(f"\nWARNING: {len(conf)} files have identical bytes but DIFFERENT labels (all excluded):")
    print(conf[["image_id", "label", "generator"]].head(10).to_string(index=False))

ok = CANDIDATES[CANDIDATES["status"] == "ok"]
near = vs_data.near_duplicate_groups(ok)
print(f"\nNear-identical groups among usable candidates (dHash within 12 of 256 bits): {near.nunique()} groups, "
      f"{int(near.notna().sum())} images. Only copies that land in DIFFERENT splits matter; Section 7 removes those.")
if near.notna().any():
    show = ok.assign(group=near).dropna(subset=["group"]).sort_values("group")
    print(show[["group", "image_id", "label", "generator"]].head(12).to_string(index=False))

**Expected output:** a status count such as `ok 1530 / duplicate 12`, examples of any duplicates, and the number of near-identical groups.

A few duplicates are normal for GenImage real images. A large number of `label_conflict` rows means a folder rule is wrong.

📤 **Send me:** the status counts and any warnings.

## 7 · Dataset splitting (one fixed leave-one-generator-out split)
**Purpose:** create **one** split that M1, the RGB baseline, M2 and Fusion will **all reuse**.

```
 seen generators ──► train 70% │ val 15% │ test_seen 15%     (per generator and class)
 unseen generator ─► test_unseen only: never in train or val
```

- The seed is fixed, so the same split comes out on every run.
- Near-identical copies in different splits are excluded, keeping the copy in the earliest split.
- **Leakage checks run every time.** If any check fails, the notebook stops.
- The split is saved to `PROJECT_DIR/data/splits/<name>/`. If it **already exists**, it is **loaded** rather than rebuilt, so every model sees exactly the same images.

✏️ **Choose `UNSEEN_GENERATOR`:** one of the AI generator names you wrote in `FOLDER_RULES`.

In [ ]:
UNSEEN_GENERATOR = "CHANGE_ME"     # <<< CHANGE: must be one of the AI generator names from FOLDER_RULES
FORCE_REBUILD_SPLIT = False        # leave False. True would replace a saved split (other models depend on it!)

gens = vs_genimage.ai_generators(CANDIDATES)
print("AI generators found:", gens)
assert UNSEEN_GENERATOR in gens, f"Set UNSEEN_GENERATOR to one of {gens}"
CONFIG["data"]["unseen_generator"] = UNSEEN_GENERATOR
SPLIT_NAME = vs_genimage.split_name(UNSEEN_GENERATOR, N_PER_CLASS, SEED)
SPLIT_DIR = Path(CONFIG["paths"]["splits_dir"]) / SPLIT_NAME
CONFIG["data"]["split_name"] = SPLIT_NAME
print("Seen generators:", [g for g in gens if g != UNSEEN_GENERATOR], "| unseen:", UNSEEN_GENERATOR)
print("Split folder:", SPLIT_DIR)

In [ ]:
if SPLIT_DIR.exists() and (SPLIT_DIR / "split_info.json").exists() and not FORCE_REBUILD_SPLIT:
    MANIFEST, SPLITS, SPLIT_INFO = vs_genimage.load_split(SPLIT_DIR, data_root=DATA_ROOT)
    same = (SPLIT_INFO["folder_rules"] == FOLDER_RULES and SPLIT_INFO["seed"] == SEED
            and SPLIT_INFO["n_per_class"] == N_PER_CLASS and SPLIT_INFO["unseen_generator"] == UNSEEN_GENERATOR)
    assert same, ("A saved split with this name was made with different FOLDER_RULES/settings. "
                  "Fix the settings, or use a different N_PER_CLASS/seed.")
    print("Loaded the EXISTING split (fingerprints verified):", SPLIT_INFO["fingerprints"])
else:
    MANIFEST = vs_genimage.make_logo_split(CANDIDATES, UNSEEN_GENERATOR, N_PER_CLASS, CONFIG["data"]["fractions"])
    MANIFEST, NEAR_REPORT = vs_genimage.drop_cross_split_near_duplicates(MANIFEST, CONFIG["data"]["drop_near_duplicates"])
    print(f"Near-identical images found across splits: {len(NEAR_REPORT)} groups "
          f"({'excluded from the later split' if CONFIG['data']['drop_near_duplicates'] else 'NOT removed'})")
    if len(NEAR_REPORT):
        print(NEAR_REPORT.head(10).to_string(index=False))
    SPLIT_INFO = vs_genimage.save_split(MANIFEST, SPLIT_DIR, {
        "split_name": SPLIT_NAME, "dataset": "genimage", "seed": SEED, "n_per_class": N_PER_CLASS,
        "unseen_generator": UNSEEN_GENERATOR, "seen_generators": [g for g in gens if g != UNSEEN_GENERATOR],
        "folder_rules": FOLDER_RULES, "ignore_folders": IGNORE_FOLDERS, "data_root": DATA_ROOT,
        "fractions": CONFIG["data"]["fractions"], "drop_near_duplicates": CONFIG["data"]["drop_near_duplicates"]})
    MANIFEST, SPLITS, SPLIT_INFO = vs_genimage.load_split(SPLIT_DIR, data_root=DATA_ROOT)
    print("Saved NEW split to", SPLIT_DIR)

In [ ]:
LEAKAGE = vs_genimage.check_leakage(MANIFEST, UNSEEN_GENERATOR, CONFIG["data"]["drop_near_duplicates"])
print(LEAKAGE.to_string(index=False))
print("\nImages per split, generator and class:")
print(vs_genimage.split_counts(MANIFEST))
print("\nExcluded / unused candidates:", MANIFEST["split"].value_counts().drop(list(vs_genimage.SPLITS), errors="ignore").to_dict())
print("\nFiles saved:", sorted(p.name for p in SPLIT_DIR.iterdir()))

**Expected output:**
- the seen and unseen generator names
- either `Saved NEW split` or `Loaded the EXISTING split`
- a leakage table where **every row says PASS**
- a counts table. With `N_PER_CLASS = 200` and 3 generators, each seen generator gives about 140 / 30 / 30 per class (train / val / test_seen), and the unseen generator gives 200 per class in `test_unseen`.
- the saved files: `manifest.csv, split_info.json, test_seen.csv, test_unseen.csv, train.csv, val.csv`

If any check says **FAIL**, the cell stops. Do not continue. Send me the table.

📤 **Send me:** the leakage table and the counts table.

## 8 · Preprocessing (shared, bias-controlled)
**Purpose:** every image, Real or AI, goes through **exactly the same steps**:

```
open → RGB → resize shorter side to 256 → centre-crop 256×256 → JPEG re-encode (quality 75–95, fixed per image)
     → cached as uint8 on the fast local disk (a copy is saved to Drive)
train only : random 224 crop + random horizontal flip   (augmentation)
val / test : centre 224 crop                            (deterministic)
→ float tensor in 0..1 → normalised INSIDE the model
```

**Why the JPEG step?** In GenImage, real photos are usually JPEG and AI images are often PNG. Re-encoding *both* with the same JPEG range removes that "file format" shortcut. 🟦 Proposal.

**Where normalisation happens:** inside each model's first step (`FeatureBuilder`), using ImageNet mean/std for M1. The cache stays un-normalised, so M2 can reuse the *same* cache with CLIP normalisation.

**Why no colour jitter or blur augmentation?** Those would destroy the faint noise and frequency traces M1 is looking for.

The first run reads every image from Drive once, which takes a few minutes. Later sessions copy the finished cache from Drive.

In [ ]:
PCFG = CONFIG["preprocess"]
CACHE = {}
CACHE_META = {}
for s in vs_genimage.SPLITS:
    local = Path(CONFIG["paths"]["cache_local"]) / SPLIT_NAME / f"{s}_{PCFG['cache_size']}.npy"
    on_drive = Path(CONFIG["paths"]["cache_drive"]) / SPLIT_NAME / f"{s}_{PCFG['cache_size']}.npy"
    CACHE_META[s] = vs_preprocess.build_cache(SPLITS[s], local, PCFG, SEED, drive_copy=on_drive,
                                              workers=CONFIG["data"]["hash_workers"])
    CACHE[s] = local
print("Caches ready:", {s: str(p) for s, p in CACHE.items()})

In [ ]:
crop, hflip, rescale = PCFG["crop_size"], PCFG["train_hflip"], PCFG.get("train_rescale_range")
DATASETS = {
    "train":       vs_preprocess.CachedImageDataset(SPLITS["train"], CACHE["train"], crop, train=True, hflip=hflip,
                                                    rescale_range=rescale),                       # augmented
    "train_eval":  vs_preprocess.CachedImageDataset(SPLITS["train"], CACHE["train"], crop, train=False),   # no augmentation
    "val":         vs_preprocess.CachedImageDataset(SPLITS["val"], CACHE["val"], crop, train=False),
    "test_seen":   vs_preprocess.CachedImageDataset(SPLITS["test_seen"], CACHE["test_seen"], crop, train=False),
    "test_unseen": vs_preprocess.CachedImageDataset(SPLITS["test_unseen"], CACHE["test_unseen"], crop, train=False),
}
BS, NW = CONFIG["train"]["active_batch_size"], CONFIG["train"]["num_workers"]
LOADERS = {
    "train":       vs_preprocess.make_loader(DATASETS["train"], BS, True, NW, SEED),
    "train_eval":  vs_preprocess.make_loader(DATASETS["train_eval"], BS, False, NW, SEED),   # for feature statistics
    "val":         vs_preprocess.make_loader(DATASETS["val"], BS, False, NW, SEED),
    "test_seen":   vs_preprocess.make_loader(DATASETS["test_seen"], BS, False, NW, SEED),
    "test_unseen": vs_preprocess.make_loader(DATASETS["test_unseen"], BS, False, NW, SEED),
}
print("Random rescale for training images:", rescale or "off")
batch = next(iter(LOADERS["train"]))
print("One training batch:", tuple(batch["x"].shape), batch["x"].dtype,
      f"values {batch['x'].min():.2f}..{batch['x'].max():.2f}", "labels", batch["label"][:8].tolist())
for s, d in DATASETS.items():
    print(f"{s:>12}: {len(d):>6} images")

### 8b · Check: random rescale changes ONLY training images
Random rescale (used by the run `m1_random_rescale`) resizes each **training** image by a random factor between ×0.5 and ×2 before the 224 crop, so every training image gets a random resampling trace that says nothing about its label. This cell checks three things and **saves nothing**:
1. augmented training images are still 3 × 224 × 224, values 0..1
2. val, test_seen and test_unseen are **identical** to the baseline preprocessing and deterministic
3. a picture: the same training images at fixed scales ×0.5 … ×2 (after the 224 crop)

In [ ]:
RESCALE_CHECK = PCFG.get("train_rescale_range") or (0.5, 2.0)
aug = vs_preprocess.CachedImageDataset(SPLITS["train"], CACHE["train"], crop, train=True, hflip=hflip,
                                       rescale_range=RESCALE_CHECK)

# 1. augmented training images keep the model's input size
random.seed(SEED)
xs = [aug[i]["x"] for i in range(min(64, len(aug)))]
print("1. augmented train shapes:", {tuple(x.shape) for x in xs},
      f"values {min(float(x.min()) for x in xs):.2f}..{max(float(x.max()) for x in xs):.2f}")

# 2. evaluation data is untouched: same tensors as a fresh baseline dataset, and the same on every read
for s in ("val", "test_seen", "test_unseen"):
    ref = vs_preprocess.CachedImageDataset(SPLITS[s], CACHE[s], crop, train=False)
    same = all(torch.equal(DATASETS[s][i]["x"], ref[i]["x"]) and torch.equal(DATASETS[s][i]["x"], DATASETS[s][i]["x"])
               for i in range(len(ref)))
    print(f"2. {s:>11}: identical to baseline + deterministic -> {'PASS' if same else 'FAIL'}")

# 3. one training image per (generator, class) at fixed scales, centre 224 crop
scales = [0.5, 0.7, 1.0, 1.4, 2.0]
tr = SPLITS["train"].reset_index(drop=True)
pick = [g.index[0] for _, g in tr.groupby(["generator", "label"])]
fig, axes = plt.subplots(len(pick), len(scales), figsize=(2.3 * len(scales), 2.5 * len(pick)))
for r, i in enumerate(pick):
    a = np.load(str(CACHE["train"]), mmap_mode="r")[i]
    for c, s in enumerate(scales):
        b = vs_preprocess.random_rescale(a, RESCALE_CHECK, crop, s=s)
        o = (b.shape[0] - crop) // 2
        axes[r][c].imshow(b[o:o + crop, o:o + crop]); axes[r][c].axis("off")
        axes[r][c].set_title(f"{tr.generator[i]} {'AI' if tr.label[i] else 'Real'} x{s}", fontsize=8)
fig.suptitle("Training images after random rescale (fixed scales shown) + 224 crop")
plt.tight_layout(); plt.show()

In [ ]:
# Bias-control evidence AFTER preprocessing
rows = []
for s in vs_genimage.SPLITS:
    df = SPLITS[s].assign(upscaled=CACHE_META[s]["upscaled"], jpeg_q=CACHE_META[s]["jpeg_quality"])
    for (g, lab), d in df.groupby(["generator", "label"]):
        rows.append({"split": s, "generator": g, "class": "ai" if lab else "real", "n": len(d),
                     "upscaled_%": round(100 * d["upscaled"].mean(), 1), "jpeg_q_mean": round(d["jpeg_q"].mean(), 1)})
BIAS = pd.DataFrame(rows)
print(BIAS.to_string(index=False))
print("\nEvery cached image is now 256x256, RGB, JPEG re-encoded with the same quality range.")
if (BIAS["upscaled_%"] > 20).any():
    print("WARNING: some groups were mostly UPSCALED (originals smaller than 256 px). Upscaling leaves its own "
          "interpolation pattern. If it affects only one class or generator, the model could learn it. Note this "
          "for the report, and consider a smaller resize_short_side for every image.")

**Expected output:**
- progress bars, then `Caches ready`
- `One training batch: (64, 3, 224, 224) torch.float32 values 0.00..1.00 labels [...]`
- dataset sizes
- a bias table: `jpeg_q_mean` should be about 85 for **both** classes, and `upscaled_%` ideally near 0 everywhere

📤 **Send me:** the batch line, the sizes, the bias table, and any WARNING.

## 9 · Data visualisation
**Purpose:** check with your own eyes:
1. that labels look right
2. what preprocessing does: before (original file) versus after (224×224, JPEG re-encoded)
3. what a training batch looks like after augmentation

In [ ]:
RUN_ROOT = vs_common.make_run_dir(Path(CONFIG["paths"]["experiments_dir"]) / RUN_NAME, resume=True)
DATA_PLOTS = RUN_ROOT / "data_plots"
DATA_PLOTS.mkdir(exist_ok=True)

tr = SPLITS["train"].reset_index().rename(columns={"index": "cache_index"})
pick = pd.concat([g.sample(min(2, len(g)), random_state=SEED) for _, g in tr.groupby(["generator", "label"])])
vs_plots.plot_before_after(pick.head(8), CACHE["train"], DATA_PLOTS / "before_after_preprocessing.png")

In [ ]:
b = next(iter(LOADERS["train"]))
n = min(12, len(b["x"]))
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, i in zip(axes.flat, range(n)):
    ax.imshow(b["x"][i].permute(1, 2, 0).numpy())
    ax.set_title("AI" if b["label"][i] == 1 else "Real", color="#1f5fbf" if b["label"][i] == 1 else "#2b8a3e")
    ax.axis("off")
fig.suptitle("A training batch after random crop + flip (what the model receives, before normalisation)")
plt.tight_layout(); plt.savefig(DATA_PLOTS / "training_batch.png", dpi=110); plt.show()

**Expected output:** two image grids, saved under `experiments/<RUN_NAME>/data_plots/`.

**Check:** do images labelled **AI** look generated, and images labelled **Real** look like photos? If not, a folder rule is wrong. Fix it in Section 5 and rebuild the split with a new `N_PER_CLASS` or `SEED`.

📤 **Send me:** screenshots of both grids.

## 10 · RGB baseline
**Purpose:** a standard ResNet-50 that looks at ordinary colour images. It is trained on the **same split, preprocessing and settings** as M1, so any difference comes from the *input features*. ⬜ addition: a controlled comparison.

```
x (0..1) → ImageNet normalisation → ResNet-50 (ImageNet weights, unchanged first layer)
         → 2048-d feature → Dropout → Linear(2048 → 1) → logit → sigmoid → P(AI)
```

The **same class** (`ResNetDetector`) builds both models. Only `input_mode` changes: `"rgb"` here, `"hp_fft"` for M1.

In [ ]:
vs_common.show_source(vs_models.ResNetDetector.forward)
rgb_model = vs_models.build_detector("rgb", CONFIG).to(DEVICE)
print("first layer:", rgb_model.backbone.conv1)
print("head:", rgb_model.head)
print(f"parameters: {vs_models.count_parameters(rgb_model):,}")
with torch.no_grad():
    out = rgb_model.eval()(batch["x"][:4].to(DEVICE))
print({k: tuple(v.shape) for k, v in out.items()})
del rgb_model

**Expected output:** the `forward` code, then `first layer: Conv2d(3, 64, kernel_size=(7, 7), ...)`, the head, about **23.5 million parameters**, and the output shapes `{'logits': (4,), 'prob': (4,), 'pred': (4,), 'features': (4, 2048)}`. (The ImageNet weights download, about 100 MB, the first time.)

## 11 · High-pass feature extraction
**How it is made:** `residual = image − GaussianBlur(image)`, using a 5×5 kernel with σ = 1, separately for each colour channel.

```
image ──► blur (keeps smooth shapes and colours) ──┐
  │                                                ▼
  └──────────────────────────────────────────► subtract ──► residual: edges, fine texture, noise
```

**Why it helps:** the residual removes most of the *picture content* and keeps fine detail and noise, where generators leave fingerprints. Residual values are tiny (about ±0.05), so M1 divides them by their typical size, `hp_std`, measured on **training images only**.

In [ ]:
vs_common.show_source(vs_features.high_pass)
fb_vis = vs_features.FeatureBuilder("rgb_hp_fft", CONFIG["features"]["hp_kernel"], CONFIG["features"]["hp_sigma"])
vis = pd.concat([g.head(1) for _, g in tr.groupby(["generator", "label"])]).head(6)
xs = torch.stack([DATASETS["train_eval"][int(i)]["x"] for i in vis["cache_index"]])
vs_plots.plot_representations(xs, vis["label"].tolist(), vis["generator"].tolist(), fb_vis,
                              DATA_PLOTS / "rgb_highpass_fft.png")

**Expected output:** the `high_pass` code, then a grid with one row per (generator, class) and three columns: **RGB**, **high-pass |residual|** (bright = strong fine detail) and **FFT log-magnitude**.

Don't expect differences you can see by eye. The model looks for statistical patterns across thousands of images.

## 12 · FFT feature extraction
**How it is made:**

```
image → grey → 2D FFT (image as a sum of waves) → |·| (strength of each wave)
      → fftshift (slow waves to the centre) → log(1 + ·) (squeeze the huge range) → 1 channel, 224×224
```

**Why it helps:** natural photos have energy that fades smoothly from the centre. Upsampling layers in many generators leave **periodic** patterns, which show up as bright **off-centre dots or lines**. The average spectrum over many images makes these easier to see than any single image.

The FFT is computed in float32 even under mixed precision, because half-precision FFTs are unreliable at size 224. Values are standardised with a mean and std measured on training images only.

In [ ]:
vs_common.show_source(vs_features.fft_log_magnitude)
spectra = {}
for (g, lab), d in tr.groupby(["generator", "label"]):
    idx = d["cache_index"].head(64).tolist()
    x = torch.stack([DATASETS["train_eval"][int(i)]["x"] for i in idx])
    spectra[(g, lab)] = vs_features.fft_log_magnitude(x).mean(0)[0].numpy()
vs_plots.plot_mean_spectra(spectra, DATA_PLOTS / "mean_fft_spectra.png")

**Expected output:** the `fft_log_magnitude` code, then one average spectrum per (generator, class) from up to 64 training images each.

**What to look for:** bright dots or grid lines in the **AI** panels that are absent from the **Real** ones. Their presence or absence is an observation for the report, not a result.

The bright **centre** and the thin **cross** through it appear in *every* panel. The centre is the image's average brightness and smooth shading. The cross comes from the image edges, which don't wrap around. Neither one is a generator artefact.

📤 **Send me:** screenshots of the Section 11 and 12 figures.

## 13 · M1 model (proposal: high-pass + FFT ResNet-50)
**Input:** 3 high-pass channels + 1 FFT channel = **4 channels**. ImageNet ResNet-50 expects 3, so the **first convolution is replaced**:
- the high-pass channels **copy** the pretrained RGB filters (they still respond to edges and texture)
- the FFT channel uses their **average**
- the weights are scaled by 3/4 so activations start at a familiar size

Everything after the first layer keeps its ImageNet weights. The 1000-class head is replaced by `Dropout → Linear(2048 → 1)`.

```
x (0..1) ─┬─ high-pass ÷ hp_std ─────────────┐
          └─ FFT log-mag, standardised ───────┴─► 4-channel input ─► new conv1 ─► ResNet-50 body
             ─► 2048-d FEATURE (kept for Fusion) ─► Dropout ─► Linear ─► logit ─► P(AI) ─► label (≥ 0.5 = AI)
```

The **forward method returns** `logits`, `prob` (P(AI)), `pred` (0 = Real, 1 = AI) and `features` (2048-d). That is the interface the future Fusion model will use.

In [ ]:
vs_common.show_source(vs_models.adapt_conv1)
m1 = vs_models.build_detector("hp_fft", CONFIG).to(DEVICE)
print("first layer:", m1.backbone.conv1)
stats = m1.features_in.fit_stats(LOADERS["train"], DEVICE, max_batches=5)
print("feature scale statistics (from training images):", {k: round(v, 5) for k, v in stats.items()})
with torch.no_grad():
    z = m1.features_in(batch["x"][:4].to(DEVICE))
    out = m1.eval()(batch["x"][:4].to(DEVICE))
print("model input:", tuple(z.shape), f"mean {z.mean():.2f} std {z.std():.2f}")
print({k: tuple(v.shape) for k, v in out.items()}, "| P(AI) before training:", out["prob"].cpu().numpy().round(3))
del m1

**Expected output:**
- `first layer: Conv2d(4, 64, kernel_size=(7, 7), ...)`
- the fitted `hp_std`, `fft_mean` and `fft_std`
- `model input: (4, 4, 224, 224)`, with mean near 0 and std near 1
- output shapes `(4,) (4,) (4,) (4, 2048)`
- P(AI) values near 0.5, because the model is untrained

## 14 · Sanity checks (must pass before training)
These cheap checks catch most beginner bugs before you spend A100 hours.

| Check | What it does | Pass if | If it fails |
|---|---|---|---|
| **Smoke test** | 2 training steps + 1 validation batch for each model | runs, outputs have the right shapes, no NaN | shape or device bug |
| **Overfit 32 images** | train repeatedly on 32 images (no augmentation) | accuracy on those 32 ≥ 0.9 | the model cannot learn even when it should: a label, input or optimiser bug |
| **Random labels (mirrored pair)** | model A trains on random labels that carry *zero* information about the true class; model B trains on the exact mirror labels (every 0↔1). Both start identically, with the last layer set to zero. Then score both on the real validation labels | the **average** of the two validation AUROCs is 0.5 ± 0.10 | the model is learning the true label from somewhere other than the labels we give it: a bug in the label path |

All three use the hp_fft M1 model. The smoke test covers both models.

**Why a mirrored pair?** When the classes are easy to separate, a *single* random-label model can land near AUROC 0 or 1 just because of its random starting weights, even with no leakage. With a zero last layer and identical randomness, model B is an exact mirror of model A whenever the pipeline learns only from the labels it is given. So AUROC_B = 1 − AUROC_A and the average is 0.5. Individual A and B values far from 0.5 are therefore normal; only the **average** matters.

This test checks the **label path**. Duplicate images across splits are checked separately, in Section 7.

In [ ]:
SANITY = {}
make = lambda mode: (lambda: vs_models.build_detector(mode, CONFIG))
for mode in CONFIG["runs"]["modes"]:
    vs_common.set_seed(SEED)
    model = make(mode)().to(DEVICE)
    model.features_in.fit_stats(LOADERS["train"], DEVICE, max_batches=2)
    opt = torch.optim.AdamW(model.param_groups(1e-4, 1e-3, 1e-4))
    t0 = time.time()
    tr_out = vs_train.train_one_epoch(model, LOADERS["train"], opt, None, vs_train.make_scaler(AMP), DEVICE, AMP, max_batches=2)
    pv, vloss = vs_train.predict(model, LOADERS["val"], DEVICE, AMP, max_batches=1)
    with torch.no_grad():
        o = model.eval()(batch["x"][:2].to(DEVICE))
    ok = (o["logits"].shape == (2,) and o["features"].shape == (2, 2048)
          and math.isfinite(tr_out["loss"]) and math.isfinite(vloss))
    SANITY[f"smoke_{mode}"] = ok
    print(f"[smoke {mode:>6}] {'PASS' if ok else 'FAIL'}  train_loss {tr_out['loss']:.3f}  val_loss {vloss:.3f}  "
          f"{(time.time() - t0) / 3:.2f}s per batch")
    del model, opt

In [ ]:
vs_common.set_seed(SEED)
n32 = min(CONFIG["sanity"]["overfit_images"], len(DATASETS["train_eval"]))
rng = np.random.default_rng(SEED)
idx32 = sorted(rng.choice(len(DATASETS["train_eval"]), size=n32, replace=False).tolist())
steps = CONFIG["sanity"]["overfit_steps_gpu"] if DEVICE.type == "cuda" else CONFIG["sanity"]["overfit_steps_cpu"]
res = vs_train.overfit_test(make("hp_fft"), torch.utils.data.Subset(DATASETS["train_eval"], idx32), DEVICE, AMP,
                            steps=steps, batch_size=min(16, n32), seed=SEED)
SANITY["overfit_32"] = res["passed"]
print(f"[overfit {n32} images] {'PASS' if res['passed'] else 'FAIL'}  accuracy {res['accuracy']:.3f}  loss {res['final_loss']:.3f}  ({steps} steps)")

In [ ]:
vs_common.set_seed(SEED)
res = vs_train.shuffled_label_test(make("hp_fft"), DATASETS["train"], LOADERS["val"], CONFIG, DEVICE, AMP, SEED)
SANITY["shuffled_labels"] = res["passed"]
lo, hi = res["band"]
print(f"[shuffled labels] {'PASS' if res['passed'] else 'FAIL'}  mean val AUROC {res['mean_auroc']:.3f} "
      f"(pass band {lo:.2f}-{hi:.2f})  | model A {res['auroc_A']:.3f}, mirror model B {res['auroc_B_mirror']:.3f}"
      f"  | {res['train_images']} images, {res['epochs']} epochs each")
print("\nSanity summary:", SANITY)
SANITY_PASSED = all(SANITY.values())
print("ALL SANITY CHECKS PASSED" if SANITY_PASSED else "SOME CHECKS FAILED: send me the output before training.")

**Expected output:**
- `[smoke rgb] PASS` and `[smoke hp_fft] PASS`, with time per batch
- `[overfit 32 images] PASS  accuracy 1.000` or close to it
- `[shuffled labels] PASS  mean val AUROC ≈ 0.5 (pass band ...) | model A ..., mirror model B ...`
- `ALL SANITY CHECKS PASSED`

Model A and B can each be far from 0.5. Only their mean is tested. If the overfit check fails on CPU, it may simply need more steps; on the A100 it should pass.

📤 **Send me:** all sanity lines.

## 15 · Training
**Purpose:** train the **RGB baseline** and **M1** one after the other, with identical settings:
- **Loss:** `BCEWithLogitsLoss`, the standard loss for one yes/no output. It combines the sigmoid and the log-loss in one numerically stable step.
- **Optimiser:** AdamW. Pretrained layers use lr 1e-4; the new layers (head, adapted conv1) use lr 1e-3. Weight decay 1e-4.
- **Learning rate:** 1 warm-up epoch, then a cosine decay.
- **Mixed precision:** on CUDA (bfloat16 on the A100).
- **After every epoch:** validation loss, accuracy and AUROC, all on **seen** generators only.
- **`best.pt`:** saved whenever validation AUROC improves.
- **`last.pt`:** saved every epoch, so you can resume after a disconnect.
- **Early stopping:** after 3 epochs without improvement.

The test sets are **not used here at all**.

Each model gets its own folder, `experiments/<RUN_NAME>/<mode>/`.

In [ ]:
assert "SANITY_PASSED" in dir() and SANITY_PASSED, "Run Section 14 first; all sanity checks must pass before training."
if DEVICE.type != "cuda":
    print("CPU detected: limiting training to a tiny smoke run (2 epochs x 3 batches). Use the A100 for real training.")
    CONFIG["train"].update(epochs=min(2, CONFIG["train"]["epochs"]), max_train_batches=3, max_val_batches=3)

RUN_DIRS, HISTORIES = {}, {}
for mode in CONFIG["runs"]["modes"]:
    print(f"\n{'=' * 22} training: {mode} {'(RGB baseline)' if mode == 'rgb' else '(M1)'} {'=' * 22}")
    run_dir = vs_common.make_run_dir(RUN_ROOT / mode, resume=RESUME)
    run_cfg = {**CONFIG, "model": {**CONFIG["model"], "input_mode": mode},
               "environment": ENV, "split_fingerprints": SPLIT_INFO["fingerprints"]}
    vs_common.save_yaml(run_cfg, run_dir / "config.yaml")
    vs_common.set_seed(SEED)
    model = vs_models.build_detector(mode, CONFIG).to(DEVICE)
    t0 = time.time()
    stats_loader = LOADERS["train_eval"] if PCFG.get("train_rescale_range") else None   # un-augmented train images
    HISTORIES[mode] = vs_train.fit(model, LOADERS["train"], LOADERS["val"], run_cfg, run_dir, DEVICE, AMP, resume=RESUME,
                                   stats_loader=stats_loader)
    RUN_DIRS[mode] = run_dir
    print(f"{mode}: finished in {(time.time() - t0) / 60:.1f} min -> {run_dir}")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

**Expected output:** for each model, the feature statistics (M1 only), then one line per epoch:

`epoch  1/10  train_loss 0.6xx  train_acc 0.6xx  val_loss ...  val_acc ...  val_auroc ...  * best  (NNs)`

**How to read it while it trains:**
- `train_loss` should go **down**.
- `val_auroc` should go **up**, then level off.
- `train_loss` still falling while `val_loss` rises is **overfitting**. Early stopping and `best.pt` protect you from it.

📤 **Send me:** the full printed log for both models.

## 16 · Validation
**Purpose:** reload the **frozen best checkpoint** (highest validation AUROC) of each model and score the validation set again. This confirms the saved file reproduces the number seen during training, and saves `predictions_val.csv`.

From here on, models are **frozen**: nothing is trained or tuned.

In [ ]:
RUN_DIRS = {m: RUN_ROOT / m for m in CONFIG["runs"]["modes"]}

def load_best(mode):
    ck = vs_common.load_checkpoint(RUN_DIRS[mode] / "best.pt", DEVICE)
    model = vs_models.build_detector(mode, CONFIG).to(DEVICE)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, ck

def evaluate_split(mode, split):
    model, ck = load_best(mode)
    pv, loss = vs_train.predict(model, LOADERS[split], DEVICE, AMP)
    preds = vs_train.predictions_with_meta(pv, SPLITS[split], split, f"{RUN_NAME}/{mode}")
    preds.to_csv(RUN_DIRS[mode] / f"predictions_{split}.csv", index=False)
    del model
    return preds, loss, ck

PREDS = {m: {} for m in CONFIG["runs"]["modes"]}
for mode in CONFIG["runs"]["modes"]:
    preds, loss, ck = evaluate_split(mode, "val")
    PREDS[mode]["val"] = preds
    m = vs_metrics.classification_metrics(preds["label"], preds["p_ai"])
    print(f"{mode:>7}: best epoch {ck['epoch']}  val AUROC {m['auroc']:.4f} (during training: {ck['val_metrics']['auroc']:.4f})  "
          f"val acc {m['accuracy']:.4f}  loss {loss:.4f}")

**Expected output:** one line per model. The reloaded val AUROC should equal the training-time value to about 3 decimals; tiny GPU rounding differences are fine.

## 17 · Seen-generator testing (in-distribution)
**Purpose:** score `test_seen`: images from the **same generators** the model trained on, but never seen during training. This is the "familiar generators" number. Comparing it with Section 18 measures generalisation.

In [ ]:
for mode in CONFIG["runs"]["modes"]:
    preds, loss, _ = evaluate_split(mode, "test_seen")
    PREDS[mode]["test_seen"] = preds
    m = vs_metrics.classification_metrics(preds["label"], preds["p_ai"])
    print(f"{mode:>7}: test_seen  n={m['n']}  acc {m['accuracy']:.4f}  AUROC {m['auroc']:.4f}")
    print(vs_metrics.metrics_by_generator(preds).round(4).to_string(index=False), "\n")

**Expected output:** overall accuracy and AUROC per model, plus a per-generator table (each seen AI generator against all real test images).

### 17b · Shortcut diagnostic: does resizing history drive the score? (seen data only)
GenImage AI images have fixed sizes (for example BigGAN 128², ADM 256², SD 512²); real photos vary. Resizing everything to 256 therefore leaves **different resampling traces per class**. This cell asks whether real photos that had to be **upscaled** get a higher P(AI) than other real photos. It uses only val and test_seen, never test_unseen.

**How to read it:** if upscaled real photos score clearly higher than non-upscaled ones, the model is using resize history as a shortcut.

In [ ]:
# Do Real photos that were UPSCALED in preprocessing look "more AI" to the models? (val + test_seen only)
SHORTCUT_DIAG = {}
for mode in PREDS:
    p = pd.concat([PREDS[mode][s].assign(upscaled=np.asarray(CACHE_META[s]["upscaled"], dtype=bool)[PREDS[mode][s]["cache_index"]])
                   for s in ("val", "test_seen")])
    r = p[p["label"] == 0]                                     # Real images only
    print(f"{mode:>6}: Real upscaled     n={int(r.upscaled.sum()):>3}  mean P(AI) {r[r.upscaled].p_ai.mean():.2f}")
    print(f"{'':>6}  Real not upscaled n={int((~r.upscaled).sum()):>3}  mean P(AI) {r[~r.upscaled].p_ai.mean():.2f}")
    for g, d in p[p["label"] == 1].groupby("generator"):      # AI images per generator
        print(f"{'':>6}  AI {g:<7} n={len(d):>3}  mean P(AI) {d.p_ai.mean():.2f}")
    SHORTCUT_DIAG[mode] = {"real_upscaled_mean_p_ai": float(r[r.upscaled].p_ai.mean()), "real_upscaled_n": int(r.upscaled.sum()),
                           "real_not_upscaled_mean_p_ai": float(r[~r.upscaled].p_ai.mean()),
                           "ai_mean_p_ai": {g: float(d.p_ai.mean()) for g, d in p[p["label"] == 1].groupby("generator")}}
    vs_common.save_json(SHORTCUT_DIAG[mode], RUN_DIRS[mode] / "shortcut_diagnostic.json")

## 18 · Unseen-generator testing
**Purpose:** score `test_unseen`: images from the generator that was **never** used for training or validation. This is the key RQ1 number.

⚠️ **Rule:** look at this only **after** all choices are final. If you change anything because of these numbers (epochs, learning rate, input mode), the unseen set is no longer "unseen" for that new run. Every evaluation is logged with a timestamp in `unseen_test_log.json` for honesty.

In [ ]:
for mode in CONFIG["runs"]["modes"]:
    log_path = RUN_DIRS[mode] / "unseen_test_log.json"
    log = vs_common.load_json(log_path) if log_path.exists() else []
    if log:
        print(f"NOTE ({mode}): the unseen test was already run {len(log)} time(s), first at {log[0]['time']}. "
              "Re-running the same frozen checkpoint gives the same numbers. Do not tune on them.")
    preds, loss, ck = evaluate_split(mode, "test_unseen")
    PREDS[mode]["test_unseen"] = preds
    log.append({"time": vs_common.now(), "checkpoint_epoch": ck["epoch"]})
    vs_common.save_json(log, log_path)
    m = vs_metrics.classification_metrics(preds["label"], preds["p_ai"])
    print(f"{mode:>7}: test_unseen ({UNSEEN_GENERATOR})  n={m['n']}  acc {m['accuracy']:.4f}  AUROC {m['auroc']:.4f}")

**Expected output:** one line per model with accuracy and AUROC on the unseen generator.

## 19 · Metrics
**Purpose:** compute every metric for val, test_seen and test_unseen, plus the **generalisation gap**, and save them to `metrics.json` for each model.

| Metric | Meaning (AI = positive class) |
|---|---|
| Accuracy | share of all images classified correctly at P(AI) ≥ 0.5 |
| AUROC | chance that a random AI image scores higher than a random real image. Needs no threshold; 0.5 = guessing |
| Precision | of the images called AI, the share that really are AI |
| Recall | of the AI images, the share that were caught |
| F1 | balance of precision and recall |
| Confusion matrix | `[[TN, FP], [FN, TP]]`: real→real, real→AI / AI→real, AI→AI |
| **Generalisation gap** | seen − unseen. **Positive = performance drops** on the unseen generator |

In [ ]:
METRICS = {}
for mode in CONFIG["runs"]["modes"]:
    out = {"model": mode, "run": RUN_NAME, "split_name": SPLIT_NAME, "unseen_generator": UNSEEN_GENERATOR,
           "best_epoch": vs_common.load_checkpoint(RUN_DIRS[mode] / "best.pt", "cpu")["epoch"]}
    for split in ("val", "test_seen", "test_unseen"):
        p = PREDS[mode][split]
        out[split] = vs_metrics.classification_metrics(p["label"], p["p_ai"])
        out[split]["by_generator"] = vs_metrics.metrics_by_generator(p).to_dict("records")
    out["generalization_gap"] = vs_metrics.generalization_gap(out["test_seen"], out["test_unseen"])
    vs_common.save_json(out, RUN_DIRS[mode] / "metrics.json")
    METRICS[mode] = out
    print(f"\n{mode}:")
    print(pd.DataFrame({s: {k: out[s][k] for k in vs_metrics.KEYS} for s in ("val", "test_seen", "test_unseen")}).round(4))
    print("confusion test_seen  ", out["test_seen"]["confusion_matrix"], "   test_unseen", out["test_unseen"]["confusion_matrix"])
    print("gap (seen - unseen):", {k: round(v, 4) for k, v in out["generalization_gap"].items()})

**Expected output:** for each model, a small table (rows: accuracy, AUROC, precision, recall, F1; columns: val, test_seen, test_unseen), the two confusion matrices and the gap values.

📤 **Send me:** this whole output.

## 20 · Plots
All plots are saved in `experiments/<RUN_NAME>/<mode>/plots/`, and comparison plots in `experiments/<RUN_NAME>/plots/`:
1. training and validation loss, plus validation AUROC and accuracy per epoch
2. ROC curves: val, seen and unseen on one chart
3. confusion matrices: seen and unseen
4. P(AI) score distributions per class: seen and unseen
5. examples of **correct** predictions (unseen test)
6. examples of **incorrect** predictions (unseen test), most confident mistakes first
7. comparison: ROC of RGB baseline versus M1 on the unseen generator

In [ ]:
for mode in CONFIG["runs"]["modes"]:
    P = RUN_DIRS[mode] / "plots"
    name = "RGB baseline" if mode == "rgb" else "M1 (high-pass + FFT)"
    h = pd.read_csv(RUN_DIRS[mode] / "history.csv")
    vs_plots.plot_history(h, f"{name}: training curves", P / "history.png")
    vs_plots.plot_roc({s: (PREDS[mode][s]["label"], PREDS[mode][s]["p_ai"]) for s in ("val", "test_seen", "test_unseen")},
                      f"{name}: ROC", P / "roc.png")
    for s in ("test_seen", "test_unseen"):
        vs_plots.plot_confusion(METRICS[mode][s]["confusion_matrix"], f"{name}\n{s}", P / f"confusion_{s}.png")
        vs_plots.plot_score_hist(PREDS[mode][s], f"{name}: P(AI) on {s}", P / f"scores_{s}.png")
    vs_plots.plot_examples(PREDS[mode]["test_unseen"], CACHE["test_unseen"], True,
                           f"{name}: correct predictions (unseen generator)", P / "examples_correct_unseen.png")
    vs_plots.plot_examples(PREDS[mode]["test_unseen"], CACHE["test_unseen"], False,
                           f"{name}: incorrect predictions (unseen generator)", P / "examples_incorrect_unseen.png")

In [ ]:
if len(CONFIG["runs"]["modes"]) > 1:
    (RUN_ROOT / "plots").mkdir(exist_ok=True)
    curves = {}
    for mode in CONFIG["runs"]["modes"]:
        for s in ("test_seen", "test_unseen"):
            curves[f"{mode} | {s}"] = (PREDS[mode][s]["label"], PREDS[mode][s]["p_ai"])
    vs_plots.plot_roc(curves, f"RGB baseline vs M1: seen vs unseen ({UNSEEN_GENERATOR})", RUN_ROOT / "plots" / "roc_comparison.png")

**Expected output:** all the figures listed above, for each model.

📤 **Send me:** screenshots of the training curves, ROC curves, score distributions and incorrect examples.

## 21 · Checkpoint saving and experiment record
**Purpose:** confirm every required file exists, write a `README.md` for each run, and add one row per model to the project-wide `experiments/experiments_log.csv`, which M2 and Fusion runs will also append to.

```
experiments/<RUN_NAME>/
├── README.md, results_summary.csv/.md, plots/roc_comparison.png, data_plots/
├── rgb/      config.yaml best.pt last.pt history.csv metrics.json predictions_val.csv
│             predictions_test_seen.csv predictions_test_unseen.csv unseen_test_log.json plots/ README.md
└── hp_fft/   (same files)
```

In [ ]:
REQUIRED = ["config.yaml", "best.pt", "last.pt", "history.csv", "metrics.json", "predictions_val.csv",
            "predictions_test_seen.csv", "predictions_test_unseen.csv", "plots"]
for mode in CONFIG["runs"]["modes"]:
    d, M = RUN_DIRS[mode], METRICS[mode]
    missing = [f for f in REQUIRED if not (d / f).exists()]
    lines = [f"# {RUN_NAME} / {mode} ({'RGB baseline' if mode == 'rgb' else 'M1: high-pass + FFT ResNet-50'})", "",
             f"- Created: {vs_common.now()}", f"- Split: `{SPLIT_NAME}` (fingerprints {SPLIT_INFO['fingerprints']})",
             f"- Unseen generator: **{UNSEEN_GENERATOR}**; seen: {SPLIT_INFO['seen_generators']}",
             f"- Best epoch (by validation AUROC): {M['best_epoch']}",
             f"- Device: {ENV.get('gpu', ENV['device'])}, torch {ENV['torch']}", "",
             "| split | accuracy | AUROC | precision | recall | F1 |", "|---|---|---|---|---|---|"]
    for s in ("val", "test_seen", "test_unseen"):
        lines.append(f"| {s} | " + " | ".join(f"{M[s][k]:.4f}" for k in vs_metrics.KEYS) + " |")
    lines += ["", f"Generalisation gap (seen - unseen): " + ", ".join(f"{k} {v:+.4f}" for k, v in M["generalization_gap"].items()),
              "", "Files: " + ", ".join(sorted(p.name for p in d.iterdir()))]
    (d / "README.md").write_text("\n".join(lines))
    vs_common.append_csv_row(Path(CONFIG["paths"]["experiments_dir"]) / "experiments_log.csv", {
        "time": vs_common.now(), "run": RUN_NAME, "model": mode, "split_name": SPLIT_NAME, "unseen_generator": UNSEEN_GENERATOR,
        "n_per_class": N_PER_CLASS, "seed": SEED, "best_epoch": M["best_epoch"], "device": ENV.get("gpu", ENV["device"]),
        **{f"val_{k}": M["val"][k] for k in ("accuracy", "auroc")},
        **{f"seen_{k}": M["test_seen"][k] for k in vs_metrics.KEYS},
        **{f"unseen_{k}": M["test_unseen"][k] for k in vs_metrics.KEYS},
        "auroc_gap": M["generalization_gap"]["auroc_gap"], "run_dir": str(d)})
    size = (d / "best.pt").stat().st_size / 1e6
    print(f"{mode:>7}: {'all files present' if not missing else 'MISSING ' + str(missing)}   best.pt {size:.0f} MB   -> {d}")

**Expected output:** `all files present` for each model, with `best.pt` about 95 MB. `experiments_log.csv` gains one row per model.

## 22 · Final results summary
**Purpose:** one table comparing the **RGB baseline versus M1** and **seen versus unseen** generators, plus the generalisation gap. It is saved as `results_summary.csv` and `.md` in `experiments/<RUN_NAME>/`.

All numbers come from the saved prediction files. Nothing is typed by hand.

In [ ]:
rows = []
for mode in CONFIG["runs"]["modes"]:
    M = METRICS[mode]
    for s, label in (("test_seen", "seen generators"), ("test_unseen", f"unseen: {UNSEEN_GENERATOR}")):
        rows.append({"model": "RGB baseline" if mode == "rgb" else ("M1 (high-pass + FFT)" if mode == "hp_fft" else mode),
                     "test set": label, "n": M[s]["n"], **{k: M[s][k] for k in vs_metrics.KEYS}})
RESULTS = pd.DataFrame(rows)
GAPS = pd.DataFrame([{"model": "RGB baseline" if m == "rgb" else ("M1 (high-pass + FFT)" if m == "hp_fft" else m),
                      **{k: METRICS[m]["generalization_gap"][f"{k}_gap"] for k in vs_metrics.KEYS}}
                     for m in CONFIG["runs"]["modes"]])
print("RESULTS (threshold 0.5 for accuracy / precision / recall / F1)")
print(RESULTS.round(4).to_string(index=False))
print("\nGENERALISATION GAP = seen - unseen  (positive = drop on the unseen generator)")
print(GAPS.round(4).to_string(index=False))
RESULTS.to_csv(RUN_ROOT / "results_summary.csv", index=False)
GAPS.to_csv(RUN_ROOT / "generalization_gap.csv", index=False)
md_lines = [f"# {RUN_NAME}: M1 results", "", f"Split `{SPLIT_NAME}`; unseen generator **{UNSEEN_GENERATOR}**; "
            f"{N_PER_CLASS} images per class per generator; seed {SEED}.", "",
            vs_common.df_to_markdown(RESULTS), "", "Generalisation gap (seen - unseen):", "",
            vs_common.df_to_markdown(GAPS)]
(RUN_ROOT / "results_summary.md").write_text("\n".join(md_lines))
(RUN_ROOT / "README.md").write_text("\n".join(md_lines + ["", "Sub-folders: " + ", ".join(CONFIG["runs"]["modes"]) + " (one per model)."]))
print("\nSaved:", RUN_ROOT / "results_summary.csv", "and .md")

**How to read the table (no conclusions until you have real numbers):**
- **AUROC on unseen** is the main RQ1 number for M1.
- **The gap** shows how much each model drops on the unfamiliar generator. Compare the RGB and M1 gaps.
- **One fold and one seed** (🟨 interim) gives a first indication, not a final answer. The proposal requires all 3 leave-one-out folds. Rerun this notebook with each generator as `UNSEEN_GENERATOR` and a new `RUN_NAME`.
- With `N_PER_CLASS = 200`, differences of a few AUROC points can be noise. Confirm with 500+ images before drawing conclusions.

### ✅ What to send back after the full run
1. Sections 4–8: counts, the leakage table, the bias table
2. Section 14: all sanity lines
3. Section 15: both training logs
4. Sections 19 and 22: the metrics and the final tables
5. Screenshots: training curves, ROC comparison, score distributions, incorrect examples

I will then check: did each step work, is there leakage, is the model learning, is it overfitting, are the results trustworthy, what to fix, and what to do next.

## 23 · Comparison: baseline M1 vs M1 + random rescale
**Purpose:** put the original M1 run (`m1_run_001`, never modified) next to this run, on the **same split**. Nothing is retrained or re-evaluated here: the cell only reads the saved `metrics.json` and prediction CSVs of both runs.

It shows:
1. AUROC, accuracy, precision, recall, F1 and the confusion matrix (TN, FP, FN, TP) for val, test_seen and test_unseen
2. the change (this run − baseline) for each metric
3. the shortcut diagnostic (as in 17b) for both runs, computed from their saved val + test_seen predictions
4. one ROC figure with both models on val and on the unseen generator

Saved in `experiments/<RUN_NAME>/comparison/`. Skipped when `RUN_NAME` is the baseline itself.

In [ ]:
BASELINE_RUN, MODE = "m1_run_001", "hp_fft"         # the original M1 run (read only)
EXP = Path(CONFIG["paths"]["experiments_dir"])
RUNS = {"baseline M1": EXP / BASELINE_RUN / MODE, f"M1 ({RUN_NAME})": EXP / RUN_NAME / MODE}

if RUN_NAME == BASELINE_RUN:
    print("RUN_NAME is the baseline itself: nothing to compare.")
else:
    for name, d in RUNS.items():
        assert (d / "metrics.json").exists(), f"{name}: {d / 'metrics.json'} not found"
    M = {name: vs_common.load_json(d / "metrics.json") for name, d in RUNS.items()}
    assert len({m["split_name"] for m in M.values()}) == 1, "The two runs used different splits: not comparable."
    print("Same split for both runs:", next(iter(M.values()))["split_name"])
    CMP_DIR = RUN_ROOT / "comparison"; CMP_DIR.mkdir(exist_ok=True)

    # 1. metrics + confusion matrix per run and split
    rows = []
    for name, m in M.items():
        for split in ("val", "test_seen", "test_unseen"):
            (tn, fp), (fn, tp) = m[split]["confusion_matrix"]
            rows.append({"model": name, "split": split, "best_epoch": m["best_epoch"],
                         **{k: round(m[split][k], 4) for k in ("auroc", "accuracy", "precision", "recall", "f1")},
                         "TN": tn, "FP": fp, "FN": fn, "TP": tp})
    COMPARISON = pd.DataFrame(rows)
    print("\n1. Metrics (threshold 0.5) and confusion matrix [TN FP / FN TP]:")
    print(COMPARISON.to_string(index=False))

    # 2. change = this run - baseline
    base, new = list(M)
    keys = ["auroc", "accuracy", "precision", "recall", "f1"]
    DELTA = (COMPARISON[COMPARISON.model == new].set_index("split")[keys]
             - COMPARISON[COMPARISON.model == base].set_index("split")[keys]).round(4)
    print(f"\n2. Change ({new} - {base}); positive = better:")
    print(DELTA.to_string())

    # 3. shortcut diagnostic for both runs (val + test_seen only)
    diag = []
    for name, d in RUNS.items():
        p = pd.concat([pd.read_csv(d / f"predictions_{s}.csv").assign(
                upscaled=lambda f, s=s: np.asarray(CACHE_META[s]["upscaled"], dtype=bool)[f["cache_index"]])
            for s in ("val", "test_seen")])
        r = p[p["label"] == 0]
        row = {"model": name, "real_upscaled_P(AI)": r[r.upscaled].p_ai.mean(), "n_upscaled": int(r.upscaled.sum()),
               "real_not_upscaled_P(AI)": r[~r.upscaled].p_ai.mean()}
        row.update({f"AI_{g}_P(AI)": dd.p_ai.mean() for g, dd in p[p["label"] == 1].groupby("generator")})
        diag.append(row)
    DIAG = pd.DataFrame(diag).round(3)
    print("\n3. Shortcut diagnostic (mean P(AI), val + test_seen):")
    print(DIAG.to_string(index=False))

    # 4. ROC: both models on val and on the unseen generator
    curves = {}
    for name, d in RUNS.items():
        for s in ("val", "test_unseen"):
            p = pd.read_csv(d / f"predictions_{s}.csv")
            curves[f"{name} | {s}"] = (p["label"], p["p_ai"])
    vs_plots.plot_roc(curves, f"Baseline M1 vs {new}  (unseen: {UNSEEN_GENERATOR})", CMP_DIR / "roc_baseline_vs_new.png")

    COMPARISON.to_csv(CMP_DIR / "comparison_baseline_vs_new.csv", index=False)
    DELTA.to_csv(CMP_DIR / "change_new_minus_baseline.csv")
    DIAG.to_csv(CMP_DIR / "shortcut_diagnostic_both.csv", index=False)
    print("\nSaved in", CMP_DIR, ":", sorted(p.name for p in CMP_DIR.iterdir()))

**How to read it:**
- **test_unseen AUROC** is the main number: did the change help on the generator the model never saw? With 400 unseen images, AUROC differences smaller than about ±0.06 (two standard errors) are within noise.
- **val** differences smaller than about ±0.1 are within noise (120 images).
- **Diagnostic:** if the shortcut was removed, upscaled and non-upscaled real photos should get similar P(AI).
- Accuracy, precision, recall and F1 depend on the 0.5 threshold. A model that simply became more cautious can gain accuracy while catching fewer AI images: check the confusion matrix.